# **Neuroimaging 2025/2026**

Project 1 - Brain morphometry and network function: structural MRI and resting
state fMRI

Created by: Albin Fredrik Wickman Lundberg (ist1119353), Ana Beatriz Machado (ist1119776), Bianca Cornaggia (ist1119620), Don Enrico Buebos Esteve (ist1118650), Jose Enrique Lopez (ist1118909) - Group 10

In this study, we will investigate how Parkinson's Disease (PD) affects brain morphometry and network function using structural MRI and resting-state fMRI (rs-fMRI).

PD is a progressive neurodegenerative disorder caused by the loss of dopaminergic neurons in the Substantia Nigra, resulting in motor symptoms such as tremors, stiffness (rigidity), and slow movement (bradykinesia).

Specifically, we will evaluate subcortical brain volume loss focusing on three regions of interest: the putamen, amygdala, and hippocampus. Neuroimaging studies have consistently reported volume reduction in these areas in PD.

The putamen shows early atrophy associated with dopamine reduction and motor impairment, the amygdala exhibits reduced volume associated with non-motor symptoms such as emotional and cognitive changes, and the hippocampus, particularly affected in later stages, demonstrates volume loss that correlates with cognitive decline and the progression towards mild cognitive impairment.

Additionally,  using rs-fMRI, we will assess the impact of PD on the Default Mode Network (DMN), as multiple studies have shown that functional connectivity alterations between the regions within this network are associated with lower cognitive performance, as well as grey and white matter abnormalities.

This analysis was performed using Python and neuroimaging libraries including Nilearn, Nibabel, and Matplotlib within the Neurodesk environment [9–12].

### **Environment set up**

In [ ]:
# Set up Neurodesk
%%capture
import os
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.environ["LD_PRELOAD"] = "";
    os.environ["APPTAINER_BINDPATH"] = "/content"
    os.environ["MPLCONFIGDIR"] = "/content/matplotlib-mpldir"
    os.environ["LMOD_CMD"] = "/usr/share/lmod/lmod/libexec/lmod"

    !curl -J -O https://raw.githubusercontent.com/neurodesk/neurocommand/main/googlecolab_setup.sh
    !chmod +x googlecolab_setup.sh
    !./googlecolab_setup.sh

    os.environ["MODULEPATH"] = ':'.join(map(str, list(map(lambda x: os.path.join(os.path.abspath('/cvmfs/neurodesk.ardc.edu.au/neurodesk-modules/'), x),os.listdir('/cvmfs/neurodesk.ardc.edu.au/neurodesk-modules/')))))


In [ ]:
# Download analysis packages
import lmod

# Set up AFNI
await lmod.load('afni/24.1.02')

# Set up FreeSurfer
await lmod.load('freesurfer/8.1.0')

# Set up FSL
await lmod.load('fsl/6.0.7.16')

In [ ]:
# Import necessary computation and visualization packages
import numpy as np
from ipyniivue import AnyNiivue
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import nibabel as nib
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi, show


### **Download dataset**

For this project, we will analyze structural MRI and rs-fMRI data from two subjects selected from a publicly available dataset. The dataset includes three groups: Healthy Controls (HC), PD patients with normal cognition (PD-NC), and PD patients with mild cognitive impairment (PD-MCI). Our analysis will focus on one subject from the HC group (sub-MJF008) and one from the PD-NC group (sub-MJF008).

In [ ]:
# Download example data for one subject from OpenNeuro
# "sub-MJF008" and "sub-MJF011" are the target subjects for this project
import os

PATTERN1 = "sub-MJF008"
PATTERN2 = "sub-MJF011"

# Ensure we are in the base /content directory
os.chdir('/content')

# Remove existing dataset directory for a clean install
! rm -rf ds005892

# Install the dataset skeleton from the GitHub repository
! datalad install https://github.com/OpenNeuroDatasets/ds005892.git

# Move into the dataset folder and download the actual data files for the specific subjects
! cd ds005892 && datalad get $PATTERN1 $PATTERN2

# Save the current path to return later if needed
original_dir = os.getcwd()

# sub-MJF008: T1w image

In [ ]:
# Change the working directory to the structural MRI data folder of the downloaded subject
os.chdir("ds005892/sub-MJF008/anat/")
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Copy standard structural images from MNI to current directory
!cp $FSLDIR/data/standard/MNI152_T1_2mm.nii.gz standard.nii.gz
!cp $FSLDIR/data/standard/MNI152_T1_2mm_brain.nii.gz standard_brain.nii.gz
!cp $FSLDIR/etc/flirtsch/T1_2_MNI152_2mm.cnf .

## 1. Dataset Contents - Current Directory

In [ ]:
# List the contents of the current directory
!ls

## 2. Dataset Images

In [ ]:
# Check T1w image info
!fslinfo sub-MJF008_T1w.nii.gz

In [ ]:
# Visualize T1w image
display(Markdown("### T1w structural image"))

MJF008_T1w = AnyNiivue()
MJF008_T1w.load_volumes([{"path": "sub-MJF008_T1w.nii.gz"}])
MJF008_T1w

Images, from left to right:

Axial slices: A (anterior) on top; P (posterior) on the bottom; L (left) on the left, R (right) on the right.
Coronal slices: S (superior) on top, I (inferior) on the bottom; L (left) on the left, R (right) on the right.
Sagittal slices: S (superior) on top, I (inferior) on the bottom; P (posterior) on the left, A (anterior) on the right.
3D rendition of the brain.

## 3. Brain extraction

FSL: Brain Extraction Tool (BET)

For skull stripping, different approaches were initially considered and tested, including AFNI's `3dSkullStrip` and FreeSurfer's `SynthStrip`, alongside FSL's `BET` (Brain Extraction Tool).

Upon visual inspection of the outputs from each method, FSL's `BET` (specifically with the applied parameters) was determined to provide the most accurate and visually appealing brain extraction for the `sub-MJF008_T1w` image. It successfully removed non-brain tissue while preserving crucial brain structures, minimizing both under- and over-stripping errors. Therefore, `BET` was selected as the preferred skull-stripping tool for this project due to its superior performance on this dataset.

In [ ]:
# Run BET
!bet sub-MJF008_T1w sub-MJF008_T1w_brain -f 0.5 # define intensity threshold = 0.5

In [ ]:
# Run BET and create a binary brain mask
!bet sub-MJF008_T1w sub-MJF008_T1w_brain.nii.gz -f 0.5 -m

Result Visualization

In [ ]:
# Visualize original T1 image (before brain extraction)
display(Markdown("### T1w structural image"))

MJF008_T1w = AnyNiivue()
MJF008_T1w.load_volumes([{"path": "sub-MJF008_T1w.nii.gz"}])
MJF008_T1w

In [ ]:
# Visualize FSL's BET results
# Threshold = 0.5
display(Markdown("### FSL BET (Thr=0.5)"))

MJF008_FSL = AnyNiivue()
MJF008_FSL.load_volumes([{"path": "sub-MJF008_T1w_brain.nii.gz"}])
MJF008_FSL

In [ ]:
# Visualize FSL's BET results
# Binary brain mask
display(Markdown("### FSL BET (Thr=0.5), Brain Mask"))

MJF008_FSL = AnyNiivue()
MJF008_FSL.load_volumes([{"path": "sub-MJF008_T1w_brain_mask.nii.gz"}])
MJF008_FSL

## 4. Tissue segmentation

FAST: tissue type segmentation and bias-field correction

In [ ]:
# Run fast (with bias field correction)
!fast -b -B sub-MJF008_T1w_brain

Outputs

Bias field corrected image: sub-MJF008_T1w_brain_restore

Bias field: sub-MJF008_T1w_brain_bias

CSF image: sub-MJF008_T1w_brain_pve_0

Grey matter image: sub-MJF008_T1w_brain_pve_1

White matter image: sub-MJF008_T1w_brain_pve_2

In [ ]:
# Visualize FSL's FAST results
# Bias field
display(Markdown("### FSL FAST - Bias field"))

MJF008_T1w_FAST_bias = AnyNiivue()
MJF008_T1w_FAST_bias.load_volumes([{"path": "sub-MJF008_T1w_brain_bias.nii.gz"}])
MJF008_T1w_FAST_bias

In [ ]:
# Visualize FSL's FAST results
# Bias field corrected T1w structural image
display(Markdown("### FSL FAST - Bias field corrected T1w image"))

MJF008_T1w_FAST_restore = AnyNiivue()
MJF008_T1w_FAST_restore.load_volumes([{"path": "sub-MJF008_T1w_brain_restore.nii.gz"}])
MJF008_T1w_FAST_restore

In [ ]:
# Visualize FSL's FAST results
# CSF
display(Markdown("### FSL FAST - CSF"))

MJF008_T1w_FAST_CSF = AnyNiivue()
MJF008_T1w_FAST_CSF.load_volumes([{"path": "sub-MJF008_T1w_brain_pve_0.nii.gz"}])
MJF008_T1w_FAST_CSF

In [ ]:
# Visualize FSL's FAST results
# Grey matter
display(Markdown("### FSL FAST - Grey matter"))

MJF008_T1w_FAST_GM = AnyNiivue()
MJF008_T1w_FAST_GM.load_volumes([{"path": "sub-MJF008_T1w_brain_pve_1.nii.gz"}])
MJF008_T1w_FAST_GM

In [ ]:
# Visualize FSL's FAST results
# White Matter
display(Markdown("### FSL FAST - White matter"))

MJF008_T1w_FAST_WM = AnyNiivue()
MJF008_T1w_FAST_WM.load_volumes([{"path": "sub-MJF008_T1w_brain_pve_2.nii.gz"}])
MJF008_T1w_FAST_WM

In [ ]:
# Visualize FSL's FAST results
# CSF, grey matter, white matter (overlayed)
display(Markdown("### FSL FAST - CSF (red), GM (blue), WM (white)"))

MJF008_T1w_FAST_all = AnyNiivue()
volumes = [{"path": "sub-MJF008_T1w_brain_pve_2.nii.gz", "colormap": "white"},
           {"path": "sub-MJF008_T1w_brain_pve_0.nii.gz", "colormap": "red"},
           {"path": "sub-MJF008_T1w_brain_pve_1.nii.gz", "colormap": "blue"}]
MJF008_T1w_FAST_all.load_volumes(volumes)
MJF008_T1w_FAST_all

After segmentation with FAST, the segmented grey matter, white matter, and CSF masks were used to estimate the total intracranial volume (ICV). This value was later used to normalize the extracted subcortical volumes, enabling more reliable anatomical comparisons.

In [ ]:
# Intracranial Volume

# Tissues volume
# CSF volume
CSF = !fslstats sub-MJF008_T1w_brain_pve_0 -V
CSF_vol = float(CSF[0].split()[1])

# GM volume
GM = !fslstats sub-MJF008_T1w_brain_pve_1 -V
GM_vol = float(GM[0].split()[1])

# WM volume
WM = !fslstats sub-MJF008_T1w_brain_pve_2 -V
WM_vol = float(WM[0].split()[1])

# ICV (Total volume)
ICV = CSF_vol + GM_vol + WM_vol

print(f"CSF: {CSF_vol} mm³")
print(f"GM: {GM_vol} mm³")
print(f"WM: {WM_vol} mm³")
print(f"ICV: {ICV} mm³")

FIRST: segmentation of sub-cortical structures

In [ ]:
# Perform segmentation of sub-cortical structures
!run_first_all -i sub-MJF008_T1w_brain -b -o MJF008_T1w_brain_segmented

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation"))

MJF008_T1w_FSL = AnyNiivue()
MJF008_T1w_FSL.load_volumes([{"path": "MJF008_T1w_brain_segmented_all_fast_firstseg.nii.gz"}])
MJF008_T1w_FSL

## 5. Volume Extraction

Volume extraction of 3 subcortical structures of interest: putamen, hippocampus, amygdala.

Since the brains of the two subjects have different dimensions, we had to perform the normalization of the volumes in order to accurately compare them, using the intracranial value we calculated before.



1. Putamen

In [ ]:
# Volume extraction of Putamen

# Left Putamen (label 12)
L_Puta = !fslstats MJF008_T1w_brain_segmented_all_fast_firstseg -l 11.5 -u 12.5 -V
L_Puta_vol = float(L_Puta[0].split()[1])

# Right Putamen (label 51)
R_Puta = !fslstats MJF008_T1w_brain_segmented_all_fast_firstseg -l 50.5 -u 51.5 -V
R_Puta_vol = float(R_Puta[0].split()[1])

# Putamen total volume
Puta = L_Puta_vol + R_Puta_vol

print(f"Left: {L_Puta_vol} mm³")
print(f"Right: {R_Puta_vol} mm³")
print(f"Puta: {Puta} mm³")

In [ ]:
# Putamen volume Normalization
Puta_norm = Puta / ICV

print(f"Puta_norm: {Puta_norm}")

Visualization

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation of Putamen (Putamen - Red and T1w - Blue)"))

# Extract Left Putamen mask (label 12)
!fslmaths MJF008_T1w_brain_segmented_all_fast_firstseg -thr 12 -uthr 12 -bin MJF008_putamen_L_Puta.nii.gz

# Extract Right Putamen mask (label 51)
!fslmaths MJF008_T1w_brain_segmented_all_fast_firstseg -thr 51 -uthr 51 -bin MJF008_putamen_R_Puta.nii.gz

MJF008_T1w_FSL = AnyNiivue()
volumes = [{"path": "MJF008_putamen_L_Puta.nii.gz", "colormap":"red"},
           {"path": "MJF008_putamen_R_Puta.nii.gz", "colormap": "red"},
           {"path": "sub-MJF008_T1w_brain.nii.gz", "colormap": "blue","opacity": 0.4}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

2. Hippocampus

In [ ]:
# Volume extraction of Hippocampus

# Left Hippocampus (label 17)
L_Hipp = !fslstats MJF008_T1w_brain_segmented_all_fast_firstseg -l 16.5 -u 17.5 -V
L_Hipp_vol = float(L_Hipp[0].split()[1])

# Right Hippocampus (label 53)
R_Hipp = !fslstats MJF008_T1w_brain_segmented_all_fast_firstseg -l 52.5 -u 53.5 -V
R_Hipp_vol = float(R_Hipp[0].split()[1])

# Hippocampus total volume
Hipp = L_Hipp_vol + R_Hipp_vol

print(f"Left: {L_Hipp_vol} mm³")
print(f"Right: {R_Hipp_vol} mm³")
print(f"Hipp: {Hipp} mm³")

In [ ]:
# Hippocampus volume Normalization
Hipp_norm = Hipp / ICV

print(f"Hipp_norm: {Hipp_norm}")

Visualization

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation of Hippocampus (Hippocampus - Red and T1w - Blue)"))

# Extract Left Hippocampus mask (label 17)
!fslmaths MJF008_T1w_brain_segmented_all_fast_firstseg -thr 17 -uthr 17 -bin MJF008_hippocampus_L_Hipp.nii.gz

# Extract Right Hippocampus mask (label 53)
!fslmaths MJF008_T1w_brain_segmented_all_fast_firstseg -thr 53 -uthr 53 -bin MJF008_hippocampus_R_Hipp.nii.gz

MJF008_T1w_FSL = AnyNiivue()
volumes = [{"path": "MJF008_hippocampus_L_Hipp.nii.gz", "colormap":"red"},
           {"path": "MJF008_hippocampus_R_Hipp.nii.gz", "colormap": "red"},
           {"path": "sub-MJF008_T1w_brain.nii.gz", "colormap": "blue","opacity": 0.4}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

3. Amygdala

In [ ]:
# Volume extraction of Amygdala

# Left Amygdala (label 18)
L_Amyg = !fslstats MJF008_T1w_brain_segmented_all_fast_firstseg -l 17.5 -u 18.5 -V
L_Amyg_vol = float(L_Amyg[0].split()[1])

# Right Amygdala (label 54)
R_Amyg = !fslstats MJF008_T1w_brain_segmented_all_fast_firstseg -l 53.5 -u 54.5 -V
R_Amyg_vol = float(R_Amyg[0].split()[1])

# Amygdala total volume
Amyg = L_Amyg_vol + R_Amyg_vol

print(f"Left: {L_Amyg_vol} mm³")
print(f"Right: {R_Amyg_vol} mm³")
print(f"Amyg: {Amyg} mm³")

In [ ]:
# Amygdala volume Normalization
Amyg_norm = Amyg / ICV

print(f"Amyg_norm: {Amyg_norm}")

Visualization

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation of Amygdala (Amygdala - Red and T1w - Blue)"))

# Extract Left Amygdala mask (label 18)
!fslmaths MJF008_T1w_brain_segmented_all_fast_firstseg -thr 18 -uthr 18 -bin MJF008_amygdala_L_Amyg.nii.gz

# Extract Right Amygdala mask (label 54)
!fslmaths MJF008_T1w_brain_segmented_all_fast_firstseg -thr 54 -uthr 54 -bin MJF008_amygdala_R_Amyg.nii.gz

MJF008_T1w_FSL = AnyNiivue()
volumes = [{"path": "MJF008_amygdala_L_Amyg.nii.gz", "colormap":"red"},
           {"path": "MJF008_amygdala_R_Amyg.nii.gz", "colormap": "red"},
           {"path": "sub-MJF008_T1w_brain.nii.gz", "colormap": "blue","opacity": 0.4}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

## 6. Registration

FNIRT: Between-subject registration

To spatially normalize the subject’s T1-weighted image to the MNI standard space (MNI152_T1), a two-step registration approach was adopted.

An initial affine linear registration was first performed using FLIRT to estimate the global transformation between the subject and template images.

This was followed by non-linear registration with FNIRT, which allowed local anatomical differences to be modeled, resulting in a more precise alignment to the standard space.

In [ ]:
# Visualize results (before registration)
display(Markdown("### Standard (blue) and T1w (red) - before registration"))

MJF008_T1w_FSL = AnyNiivue()
volumes =  [{"path": "standard_brain.nii.gz", "colormap": "blue"},
           {"path": "sub-MJF008_T1w_brain.nii.gz", "colormap": "red"}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

In [ ]:
# It's common to use FLIRT to obtain an initial approximation of the transformation, which we can later input in FNIRT
# Use an affine linear transformation with 12 DOF
!flirt -in sub-MJF008_T1w_brain.nii.gz -ref standard_brain -dof 12 -out sub-MJF008_T1wtoMNIlin -omat sub-MJF008_T1wtoMNIlin.mat

# Provide the affine transformation matrix estimated in the step above to initialize the non-linear registration
!fnirt --in=sub-MJF008_T1w_brain.nii.gz --aff=sub-MJF008_T1wtoMNIlin.mat --config=T1_2_MNI152_2mm.cnf --iout=sub-MJF008_T1wtoMNInonlin --cout=sub-MJF008_T1wtoMNI_coef --fout=sub-MJF008_T1wtoMNI_warp

In [ ]:
# Visualize results (after registration)
display(Markdown("### Standard MNI (blue) and T1w (red) - after linear registration"))

MJF008_T1w_FNIRT = AnyNiivue()
volumes =  [{"path": "sub-MJF008_T1wtoMNIlin.nii.gz", "colormap": "blue"},
           {"path": "standard_brain.nii.gz", "colormap": "red"}]
MJF008_T1w_FNIRT.load_volumes(volumes)
MJF008_T1w_FNIRT

In [ ]:
# Visualize results (after registration)
display(Markdown("### Standard MNI (blue) and T1w (red) - after non linear registration"))

MJF008_T1w_FNIRT = AnyNiivue()
volumes =  [{"path": "sub-MJF008_T1wtoMNInonlin.nii.gz", "colormap": "blue"},
           {"path": "standard_brain.nii.gz", "colormap": "red"}]
MJF008_T1w_FNIRT.load_volumes(volumes)
MJF008_T1w_FNIRT

# sub-MJF008: fMRI image

## 1. Change Directory

In [ ]:
# Change the working directory to the fieldmap data folder of the downloaded subject
import os

# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_fmap_dir = os.path.join("ds005892", PATTERN1, "fmap")

# Change to the target directory
os.chdir(target_fmap_dir)

print(f"Current directory: {os.getcwd()}")

Dataset Contents - Current Directory

In [ ]:
# List the contents of the current directory
!ls

## 2. Preprocessing of fMRI data

 ### Prepare Fieldmap

The fieldmap is a calibration image of B0, used in particular to identify and correct distortions. Unit: rads/s (precession frequency)

In [ ]:
# Obtain the delta echo time and the dwell time from the magnitude1 .json file
!cat sub-MJF008_magnitude1.json

In [ ]:
# Assign delta echo time and dwell time variables
delta_echo_time = 2.46 # ms
dwell_time = 0.008 # s

In [ ]:
# Skull-strip the fieldmap magnitude image (magnitude2 is superfluous and therefore ignored)
!bet sub-MJF008_magnitude1 sub-MJF008_magnitude1_brain

# Erode (shaving off voxels from the edges) the skull-stripped fieldmap magnitude image
!fslmaths sub-MJF008_magnitude1_brain -ero sub-MJF008_magnitude1_brain_eroded

Quality control

In [ ]:
# Visualize the fieldmap magnitude images - before erosion
display(Markdown("### Fieldmap magnitude image - before erosion "))

nv_fmap_mag = AnyNiivue()
nv_fmap_mag.load_volumes([{"path": "sub-MJF008_magnitude1_brain.nii.gz"}])
nv_fmap_mag


In [ ]:
# Visualize the fieldmap magnitude images - after erosion
display(Markdown("### Fieldmap magnitude image - after erosion "))

nv_fmap_mag_ero = AnyNiivue()
nv_fmap_mag_ero.load_volumes([{"path": "sub-MJF008_magnitude1_brain_eroded.nii.gz"}])
nv_fmap_mag_ero

In [ ]:
# Visualize the original fieldmap phase difference image
display(Markdown("### Fieldmap phase difference image "))

nv_fmap_phase = AnyNiivue()
nv_fmap_phase.load_volumes([{"path": "sub-MJF008_phasediff.nii.gz"}])
nv_fmap_phase

Fieldmap preparation and quality control

In [ ]:
# Convert the raw phase difference image into a fieldmap in radians/second,
# using the delta_echo_time (TE2-TE1) to calibrate the phase to frequency conversion

# fsl_prepare_fieldmap <scanner> <phase_image> <magnitude_image> <out_image> <deltaTE (in ms)>
!fsl_prepare_fieldmap SIEMENS sub-MJF008_phasediff sub-MJF008_magnitude1_brain_eroded sub-MJF008_rads $delta_echo_time

In [ ]:
# Visualize preprocessed fieldmap
display(Markdown("### Preprocessed fieldmap "))

nv_fmap = AnyNiivue()
nv_fmap.load_volumes([{"path": "sub-MJF008_rads.nii.gz"}])
nv_fmap

### Distortion Correction and Registration

Change Directory

In [ ]:
# Change the working directory to the rs-fMRI data folder of the downloaded subject
import os

# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_func_dir = os.path.join("ds005892", PATTERN1, "func")

# Change to the target directory
os.chdir(target_func_dir)

print(f"Current directory: {os.getcwd()}")

In [ ]:
# Visualize the functional image (one volume)
display(Markdown("### Functional Image (One Volume)"))

MJF008_fmri = AnyNiivue()
MJF008_fmri.load_volumes([{"path": "sub-MJF008_task-rest_bold.nii.gz"}])
MJF008_fmri

Dataset Contents - Current Directory

In [ ]:
# List the contents of the current directory
!ls

In [ ]:
# Obtain the phase encoding direction for epi_reg
# -j is anterior to posterior (-y)

!cat sub-MJF008_task-rest_bold.json

Dataset Images

In [ ]:
echo_time_func = 0.029
dwell_time_func = 0.000590001

In [ ]:
# Check rs-fMRI image info, especially total number of volumes (dim4)
!fslinfo sub-MJF008_task-rest_bold.nii.gz

Obtain a 3D image from the 4D functional data (for registration)

In [ ]:
# Use the middle volume as a representative single image for registration
tstart = 100 # Middle volume
tsize = 1

# Extract the middle volume
!fslroi sub-MJF008_task-rest_bold  sub-MJF008_task-rest_bold_midvol $tstart $tsize

# Skull-strip the extracted middle volume
!bet sub-MJF008_task-rest_bold_midvol sub-MJF008_task-rest_bold_midvol_brain

In [ ]:
# Visualize the functional image (middle volume - brain)
display(Markdown("### Functional Image (Middle Volume - Brain)"))

MJF008_fmri_midvol_brain = AnyNiivue()
MJF008_fmri_midvol_brain.load_volumes([{"path": "sub-MJF008_task-rest_bold_midvol_brain.nii.gz"}])
MJF008_fmri_midvol_brain

Distortion correction and registration to structural space

In [ ]:
# Change the working directory to the downloaded subject
import os

# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_dir = os.path.join("ds005892", PATTERN1)

# Change to the target directory
os.chdir(target_dir)

print(f"Current directory: {os.getcwd()}")

In [ ]:
# epi_reg inputs (use file names):
# --epi: Middle fMRI volume (brain extracted)
# --t1: Wholehead T1 image
# --t1brain: Brain extracted T1 image (not mask)
# --out: Output name
# --fmap: Fieldmap image (in rad/s)
# --fmapmag: Fieldmap magnitude image - wholehead extracted
# --fmapmagbrain: Fieldmap magnitude image - brain extracted
# --echospacing: Dwell time (s)
# --pedir: Phase encoding direction

# Main outputs:
# rest2struct.nii.gz: EPI (distortion-corrected) in T1 space
# rest2struct.mat: EPI → T1 transform
# rest2struct_warp.nii.gz: Nonlinear distortion warp (fmap)

import os

# Ensure we are in the correct subject directory for epi_reg
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN1))
print(f"Current directory before epi_reg: {os.getcwd()}")

# Create a folder for registration files
!mkdir -p reg

# Define absolute paths for all inputs
epi_input = os.path.join('func', 'sub-MJF008_task-rest_bold_midvol_brain')
t1_input = os.path.join('anat', 'sub-MJF008_T1w')
t1brain_input = os.path.join('anat', 'sub-MJF008_T1w_brain')
fmap_input = os.path.join('fmap', 'sub-MJF008_rads')
fmapmag_input = os.path.join('fmap', 'sub-MJF008_magnitude1')
fmapmagbrain_input = os.path.join('fmap', 'sub-MJF008_magnitude1_brain_eroded')
output_prefix = os.path.join('reg', 'rest2struct')

print("Verifying input file paths:")
print(f"EPI: {os.path.exists(epi_input + '.nii.gz')}")
print(f"T1: {os.path.exists(t1_input + '.nii.gz')}")
print(f"T1 Brain: {os.path.exists(t1brain_input + '.nii.gz')}")
print(f"Fmap: {os.path.exists(fmap_input + '.nii.gz')}")
print(f"Fmap Mag: {os.path.exists(fmapmag_input + '.nii.gz')}")
print(f"Fmap Mag Brain: {os.path.exists(fmapmagbrain_input + '.nii.gz')}")

!epi_reg --epi={epi_input} --t1={t1_input} --t1brain={t1brain_input} --out={output_prefix} --fmap={fmap_input} --fmapmag={fmapmag_input} --fmapmagbrain={fmapmagbrain_input} --echospacing=$dwell_time_func --pedir=-y --noclean

In [ ]:
# Obtain transformation matrix from strucutral space to undistorted functional space
# !convert_xfm -options <output> <input>
# struct2rest.mat: T1 → EPI transform (linear)
!convert_xfm -inverse -omat reg/struct2rest.mat reg/rest2struct.mat

# Unwarp example_func in structural space (rest2struct) to obtain undistorted functional image (rest_bold_midvol_brain_undistorted)
# Takes the corrected functional image that epi_reg placed in structural space (rest2struct) and brings it back into functional space using the inverse transform
!applywarp -i func/sub-MJF008_task-rest_bold_midvol_brain -r func/sub-MJF008_task-rest_bold_midvol_brain --premat=reg/rest2struct.mat --postmat=reg/struct2rest.mat -o func/sub-MJF008_task-rest_bold_midvol_brain_undistorted

Quality control of distortion correction and registration

In [ ]:
# Visualize the results WITHOUT distortion correction
display(Markdown("### B0 unwarping of functional volume: before"))

nv_distorted = AnyNiivue()
nv_distorted.load_volumes([{"path": "func/sub-MJF008_task-rest_bold_midvol_brain.nii.gz"}])
nv_distorted


In [ ]:
# Visualize the results WITH distortion correction
display(Markdown("### B0 unwarping of functional volume: after"))

nv_undistorted = AnyNiivue()
nv_undistorted.load_volumes([{"path": "func/sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz"}])
nv_undistorted

In [ ]:
# For registration comparison: register the distorted to structural image
!flirt -ref anat/sub-MJF008_T1w -in func/sub-MJF008_task-rest_bold_midvol_brain -out func/sub-MJF008_task-rest_bold_midvol_brain_distorted -applyxfm -init reg/rest2struct.mat -interp trilinear


In [ ]:
# Visualize the results of registration (EPI -> T1 -> EPI): WITHOUT DISTORTION CORRECTION
display(Markdown("### Registration of functional (red) to structural (blue) space: without distortion correction"))

nv_reg_func2highres_distorted = AnyNiivue()
volumes =  [{"path": "anat/sub-MJF008_T1w.nii.gz", "colormap": "blue"},
           {"path": "func/sub-MJF008_task-rest_bold_midvol_brain_distorted.nii.gz", "colormap": "red"}]
nv_reg_func2highres_distorted.load_volumes(volumes)
nv_reg_func2highres_distorted


In [ ]:
# Visualize the results of registration (EPI -> T1 -> EPI): WITH DISTORTION CORRECTION
display(Markdown("### Registration of functional (red) to structural (blue) space: WITH distortion correction"))

nv_reg_func2highres_undistorted = AnyNiivue()
volumes =  [{"path": "anat/sub-MJF008_T1w.nii.gz", "colormap": "blue"},
           {"path": "func/sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz", "colormap": "red"}]
nv_reg_func2highres_undistorted.load_volumes(volumes)
nv_reg_func2highres_undistorted

In [ ]:
# Visualize the results of registration (EPI -> T1): WITH DISTORTION CORRECTION
display(Markdown("### Registration of functional (red) to structural (blue) space: with distortion correction"))

nv_reg_func2highres_pos = AnyNiivue()
volumes =  [{"path": "anat/sub-MJF008_T1w.nii.gz", "colormap": "blue"},
           {"path": "reg/rest2struct.nii.gz", "colormap": "red"}]
nv_reg_func2highres_pos.load_volumes(volumes)
nv_reg_func2highres_pos

Perform registration to standard (MNI) space

In [ ]:
# Register highres space to standard space
!flirt -in anat/sub-MJF008_T1w_brain -ref anat/standard_brain -dof 12 -out reg/struct2standard -omat reg/struct2standard.mat

# Register functional space (already in structural space) to standard space
!flirt -in reg/rest2struct.nii.gz -ref anat/standard_brain -applyxfm -init reg/struct2standard.mat -out reg/func2standard


In [ ]:
task_rest_bold = nib.load('/content/ds005892/sub-MJF008/func/sub-MJF008_task-rest_bold.nii.gz')
print(task_rest_bold.affine)

In [ ]:
rest2struct = nib.load('/content/ds005892/sub-MJF008/reg/rest2struct.nii.gz')
print(rest2struct.affine)

In [ ]:
# Visualize the results of registration: functional to standard space
# Before
display(Markdown("### Registration of functional (red) to standard (blue) space: before"))

nv_reg_func2std_pre = AnyNiivue()
volumes =  [{"path": "anat/standard_brain.nii.gz", "colormap": "blue"},
           {"path": "func/sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz", "colormap": "red"}]
nv_reg_func2std_pre.load_volumes(volumes)
nv_reg_func2std_pre

In [ ]:
# Visualize the results of registration: functional to standard space
# After
display(Markdown("### Registration of functional (red) to standard (blue) space: after"))

nv_reg_func2std_pos = AnyNiivue()
volumes =  [{"path": "anat/standard_brain.nii.gz", "colormap": "blue"},
           {"path": "reg/func2standard.nii.gz", "colormap": "red"}]
nv_reg_func2std_pos.load_volumes(volumes)
nv_reg_func2std_pos

### Motion Correction

Using MCFLIRT, which loads the entire time-series and defaults to the middle (100) volume as initial template, it finds the transformation to the adjacent volume and uses this as as estimate for the next transformation.

Output: estimation of 6 motion realignment parameters: 3 translations and 3 rotations

In [ ]:
import os

# Ensure we are in the correct directory for functional data
os.chdir(os.path.join(original_dir, "ds005892", PATTERN1, "func"))

# Create folder for motion correction files
!mkdir -p mc

# --- Debugging: Check input file presence ---
print("Contents of current directory before mcflirt:")
!ls

# Run motion correction
!mcflirt -in sub-MJF008_task-rest_bold -mats -plots -rmsrel -rmsabs -spline_final

# --- Debugging: Check output files generated by mcflirt ---
print("Contents of current directory after mcflirt:")
!ls

In [ ]:
# MCFLIRT estimated rotations in radians (three)
!fsl_tsplot -i sub-MJF008_task-rest_bold_mcf.par -t 'MCFLIRT-estimated rotations (radians)' -u 1 --start=1 --finish=3 -a x,y,z -w 640 -h 144 -o rot.png

# MCFLIRT estimated translations in mm (three)
!fsl_tsplot -i sub-MJF008_task-rest_bold_mcf.par -t 'MCFLIRT-estimated translations (mm)' -u 1 --start=4 --finish=6 -a x,y,z -w 640 -h 144 -o trans.png

# MCFLIRT estimated mean displacement in mm
!fsl_tsplot -i sub-MJF008_task-rest_bold_mcf_abs.rms,sub-MJF008_task-rest_bold_mcf_rel.rms -t 'MCFLIRT-estimated mean displacement (mm)' -u 1 -w 640 -h 144 -a absolute,relative -o disp.png

Quality Control: Motion correction

In [ ]:
from IPython.display import Image
Image(filename='rot.png')

# Relatively small values, indicates gradual drift

In [ ]:
Image(filename='trans.png')

# Relatively small values

In [ ]:
Image(filename='disp.png')

# Typical

In [ ]:
# FIRST, change to the 'func' directory where the mcflirt outputs are located
os.chdir('/content/ds005892/sub-MJF008/func')

# Ensure the target mc directory exists within func/
!mkdir -p mc

# Clean up any previous partial directory if it exists to avoid 'Directory not empty' error
!rm -rf mc/sub-MJF008_task-rest_bold_mcf.mat

# Move the mcflirt-generated .mat directory into mc/
# Now operating within func/, so source paths don't need 'func/' prefix
!mv -f sub-MJF008_task-rest_bold_mcf.mat mc/

# Move the other mcflirt-generated files and plots into mc/
!mv -f sub-MJF008_task-rest_bold_mcf.par \
        sub-MJF008_task-rest_bold_mcf_abs.rms \
        sub-MJF008_task-rest_bold_mcf_abs_mean.rms \
        sub-MJF008_task-rest_bold_mcf_rel.rms \
        sub-MJF008_task-rest_bold_mcf_rel_mean.rms \
        rot.png trans.png disp.png \
        mc/

# Concatenate all 6 estimated realignment parameters in epi_mcf.cat
!cat mc/sub-MJF008_task-rest_bold_mcf.mat/MAT* > mc/sub-MJF008_task-rest_bold_mcf.cat

In [ ]:
# Generate carpet plot to visualize motion artifacts
from nilearn import plotting
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
plotting.plot_carpet(
    'sub-MJF008_task-rest_bold.nii.gz',
    figure=fig,
    axes=ax,
    title="Carpet Plot",
    standardize='zscore_sample',
    t_r=None
)

plt.show()

### Unwarp and skull-strip functional data

In [ ]:
# Unwarp 4D functional data: motion + distortion correction (functional space)
# Inputs:
# -i: raw 4D BOLD - motion corrected
# -r:
# --interp: Changed from "spline" to "trilinear" (OOM issues)

import os

# Define absolute paths for all files
subject_path = os.path.join(original_dir, 'ds005892', PATTERN1)
func_path = os.path.join(subject_path, 'func')
reg_path = os.path.join(subject_path, 'reg')
mc_path = os.path.join(func_path, 'mc')

input_image = os.path.join(func_path, 'sub-MJF008_task-rest_bold_mcf.nii.gz')
ref_image = os.path.join(func_path, 'sub-MJF008_task-rest_bold_midvol_brain_undistorted')
output_image = os.path.join(func_path, 'sub-MJF008_task-rest_bold_unwarp')
premat_file = os.path.join(mc_path, 'sub-MJF008_task-rest_bold_mcf.cat')
warp_file = os.path.join(reg_path, 'rest2struct_warp.nii.gz')
postmat_file = os.path.join(reg_path, 'struct2rest.mat')

!applywarp -i {input_image} -r {ref_image} -o {output_image} --premat={premat_file} --warp={warp_file} --postmat={postmat_file} --rel --interp=spline --paddingsize=1

In [ ]:
import os

# Check for struct2rest.mat
print(f"Checking for struct2rest.mat: {os.path.exists('/content/ds005892/sub-MJF008/reg/struct2rest.mat')}")
!ls -l /content/ds005892/sub-MJF008/reg/

# Check for sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz
print(f"Checking for sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz: {os.path.exists('/content/ds005892/sub-MJF008/func/sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz')}")
!ls -l /content/ds005892/sub-MJF008/func/

In [ ]:
# Skull-strip 4D functional data
# BET expects a 3D image so we need to compute the temporal mean, run BET on that single average volume to create a brain mask, then apply that mask to the full 4D data

# Change directory to subject/func
os.chdir('/content/ds005892/sub-MJF008/func')

# Extract temporal average of 4D data
!fslmaths sub-MJF008_task-rest_bold_unwarp -Tmean sub-MJF008_task-rest_bold_unwarp_mean

# Perform brain extraction on averaged 4D (now 3D) data and create brain mask (mask)
!bet2 sub-MJF008_task-rest_bold_unwarp_mean mask -f 0.3 -n -m; immv mask_mask mask

# Create skull-stripped time-series (epi_bet) with mask
!fslmaths sub-MJF008_task-rest_bold_unwarp -mas mask sub-MJF008_task-rest_bold_unwarp_brain

Quality control: Unwarping and skull-stripping of the entire functional volume

In [ ]:
# Visualize the results of B0 unwarping and brain extraction on a volume of the functional data
# Before
from IPython.display import display, Markdown
display(Markdown("### Unwarping and brain extraction: before"))

nv_unwarp_bet_pre = AnyNiivue()
nv_unwarp_bet_pre.load_volumes([{"path": "sub-MJF008_task-rest_bold_mcf.nii.gz"}])
nv_unwarp_bet_pre

In [ ]:
# Visualize the results of B0 unwarping and brain extraction on a volume of the functional data
# After
display(Markdown("### Unwarping and brain extraction: after"))

nv_unwarp_bet_pos = AnyNiivue()
nv_unwarp_bet_pos.load_volumes([{"path": "sub-MJF008_task-rest_bold_unwarp_brain.nii.gz"}])
nv_unwarp_bet_pos

### High-pass Temporal Filtering

In [ ]:
# Check TR
!fslinfo sub-MJF008_task-rest_bold_unwarp_brain

In [ ]:
# Define hyperparameters
TR = 2 # s
hp_freq = 0.01 # Hz
hp_sigma = 1 / (2 * TR * hp_freq)

# Mean of brain-extracted data
!fslmaths sub-MJF008_task-rest_bold_unwarp_brain -Tmean sub-MJF008_task-rest_bold_unwarp_brain_mean

# Apply high-pass filter
!fslmaths sub-MJF008_task-rest_bold_unwarp_brain -bptf {hp_sigma} -1 -add sub-MJF008_task-rest_bold_unwarp_brain_mean sub-MJF008_task-rest_bold_unwarp_brain_hpf

Quality control: High-pass filtering

In [ ]:
# Visualize the results of high pass filtering on a volume of the functional data
# Before
display(Markdown("### High pass temporal filtering of functional data: before"))

nv_hpf_pre = AnyNiivue()
nv_hpf_pre.load_volumes([{"path": "sub-MJF008_task-rest_bold_unwarp_brain.nii.gz"}])
nv_hpf_pre

In [ ]:
# Visualize the results of high pass filtering on a volume of the functional data
# AFTER
display(Markdown("### High pass temporal filtering of functional data: after"))

nv_hpf_post = AnyNiivue()
nv_hpf_post.load_volumes([{"path": "sub-MJF008_task-rest_bold_unwarp_brain_hpf.nii.gz"}])
nv_hpf_post

### Spatially smooth functional data

In [ ]:
# Check voxel size - anisotropic
!fslinfo sub-MJF008_task-rest_bold_unwarp_brain_hpf

In [ ]:
# Obtain geometric mean of voxel size (as a compromise) and use it as basis for fwhm
geom_mean_voxel = np.cbrt(3.125 * 3.125 * 3.5)

# Define hyperparameters
fwhm = 1.5 * geom_mean_voxel
sigma = fwhm / 2.355

In [ ]:
# Calculate the median brain intensity (50th percentile) within the brain mask
# Use NON-HPF data for brightness threshold (stable intensities)
median=!fslstats sub-MJF008_task-rest_bold_unwarp_brain -k mask -p 50

# Set brightness threshold at 75% of median — voxels further apart than this will not be averaged together
bright_thr = float(median[0]) * 0.75

# Same calculation for the reference image (temporal mean)
# Use NON-HPF mean as USAN reference (standard practice)
median_usan=!fslstats sub-MJF008_task-rest_bold_unwarp_brain_mean -k mask -p 50
bright_thr_usan = float(median_usan[0]) * 0.75

# Run SUSAN smoothing: edge-preserving Gaussian blur using mean_func as reference# Run SUSAN smoothing: edge-preserving Gaussian blur using mean_func as reference
# Use HPF data
!susan sub-MJF008_task-rest_bold_unwarp_brain_hpf $bright_thr $sigma 3 1 1 sub-MJF008_task-rest_bold_unwarp_brain_mean $bright_thr_usan sub-MJF008_task-rest_bold_unwarp_brain_hpf_smooth

# Reapply brain mask to remove any signal smoothed outside the brain boundary
!fslmaths sub-MJF008_task-rest_bold_unwarp_brain_hpf_smooth -mas mask sub-MJF008_task-rest_bold_unwarp_brain_hpf_smooth.nii.gz

In [ ]:
# Visualize the results of spatial smoothing on a volume of the functional data
# Before
display(Markdown("### Spatial smoothing of functional data: before"))

nv_smoothing_bet_pre = AnyNiivue()
nv_smoothing_bet_pre.load_volumes([{"path": "sub-MJF008_task-rest_bold_unwarp_brain_hpf.nii.gz"}])
nv_smoothing_bet_pre

In [ ]:
# Visualize the results of spatial smoothing on a volume of the functional data
# After
display(Markdown("### Spatial smoothing of functional data: after"))

nv_smoothing_pos = AnyNiivue()
nv_smoothing_pos.load_volumes([{"path": "sub-MJF008_task-rest_bold_unwarp_brain_hpf_smooth.nii.gz"}])
nv_smoothing_pos

### Perform ICA

In [ ]:
# Input: 4D images that have been:
# 1. Distortion corrected
# 2. Registered to structural space
# 3. Motion corrected
# 4. Skull-stripped
# 5. High-pass temporal filtered
# 6. Spatially-smoothed

# Output: melodic_IC.nii.gz
# 4D image where each volume = one independent component (IC)

# Move to subject folder
os.chdir('/content/ds005892/sub-MJF008')

# Run MELODIC
!melodic -i func/sub-MJF008_task-rest_bold_unwarp_brain_hpf_smooth -o melodic_output.ica --nobet --tr=$TR --report

### Classify and remove noise components from single-subject ICA

In [ ]:
# Create subdirectories inside melodic output to mimic FEAT structure
!mkdir -p melodic_output.ica/filtered_func_data.ica
!mkdir -p melodic_output.ica/reg
!mkdir -p melodic_output.ica/mc

# Copy ICA files into the subdirectory
!cp melodic_output.ica/melodic_* melodic_output.ica/filtered_func_data.ica/

# Copy preprocessed 4D data into the subdirectory. Rename the 4D file as filtered_func_data
!cp func/sub-MJF008_task-rest_bold_unwarp_brain_hpf_smooth.nii.gz melodic_output.ica/filtered_func_data.nii.gz

# Copy necessary supporting files and rename based of MELODIC/FEAT convention (FIX/UserGuide)
!cp anat/sub-MJF008_T1w_brain.nii.gz melodic_output.ica/reg/highres.nii.gz  # brain-extracted structural
!cp reg/struct2rest.mat melodic_output.ica/reg/highres2example_func.mat     # FLIRT transform from structural to functional space
!cp func/mc/sub-MJF008_task-rest_bold_mcf.par melodic_output.ica/mc/prefiltered_func_data_mcf.par # motion parameters created by mcflirt
!cp func/sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz melodic_output.ica/reg/example_func.nii.gz # Middle volume of 4D data as example

# Remain commented these lines
# !cp reg/rest2struct.mat melodic_output.ica/reg/example_func2highres.mat
# !cp melodic_output.ica/mask.nii.gz melodic_output.ica/filtered_func_data.ica/mask.nii.gz
# !cp melodic_output.ica/mean.nii.gz melodic_output.ica/filtered_func_data.ica/mean_func.nii.gz   # 3D image (temporal mean of 4D)
# !cp /content/ds005892/sub-MJF008/anat/standard.nii.gz melodic_output.ica/reg
# !cp sub-MJF008_task-rest_bold_midvol_brain_undistorted.nii.gz melodic_output.ica/reg/example_func.nii.gz

# Feature extraction (required for classification)
# Output: melodic_output.ica/fix
!fix -f melodic_output.ica

# Classification with pretrained classifier (Standard) with a usual threshold
# P(noise) > 20 - remove component
# Output: fix4melview_Standard_thr20.txt
!fix -c melodic_output.ica Standard 20

# Change directory to melodic output and inspect the classification results
os.chdir('/content/ds005892/sub-MJF008/melodic_output.ica')
!cat fix4melview_Standard_thr20.txt

### Nuisance Regression

Motion Outliers

In [ ]:
# Use the skull-stripped time-series as input to identify motion outliers
# Use RMS intensity difference to reference volume as metric
!fsl_motion_outliers -i filtered_func_data -o mo_regressors.txt --refrms --nomoco -m mask -s mc/refrms.txt -p mc/refrms

# Rows (volumes), columns (type of MO), 1 = presence of MO

In [ ]:
# Check the contents of the confound matrix created
# Each column represents one motion outlier timepoint
# Each row represents one volume — a value of 1 flags that volume as corrupted by large motion
!cat mo_regressors.txt

WM and CSF Signals

In [ ]:
# Change to subject directory
os.chdir('/content/ds005892/sub-MJF008')

# Run FAST to segment the structural image into CSF, GM and WM
#-b: output bias field, -B: apply bias field correction
!fast -b -B anat/sub-MJF008_T1w_brain.nii.gz

# Threshold tissue probability maps at 0.5 to create binary masks
# highres_brain_pve_0 = CSF, highres_brain_pve_1 = GM, highres_brain_pve_2 = WM
!fslmaths anat/sub-MJF008_T1w_brain_pve_0 -thr 0.5 -bin melodic_output.ica/CSF_thr
!fslmaths anat/sub-MJF008_T1w_brain_pve_2 -thr 0.5 -bin melodic_output.ica/WM_thr

# Change to melodic output directory
os.chdir('/content/ds005892/sub-MJF008/melodic_output.ica')

# Co-register CSF and WM masks from structural space into functional space
!flirt -in CSF_thr -ref reg/example_func -applyxfm -init reg/highres2example_func.mat -interp nearestneighbour -out EF_CSF_thr
!flirt -in WM_thr -ref reg/example_func -applyxfm -init reg/highres2example_func.mat -interp nearestneighbour -out EF_WM_thr

# Erode masks to avoid partial volume effects at tissue boundaries
!fslmaths EF_CSF_thr -kernel gauss 1.8 -ero -bin EF_CSF_ero
!fslmaths EF_WM_thr -kernel gauss 2.2 -ero -bin EF_WM_ero

# Extract mean timeseries from CSF and WM masks
!fslmeants -i filtered_func_data -m EF_CSF_ero -o CSF.txt
!fslmeants -i filtered_func_data -m EF_WM_ero -o WM.txt

In [ ]:
# Check the contents of the WM mean signal timeseries — one value per volume representing
# the average BOLD signal within the white matter mask at each timepoint
!cat WM.txt

Design Matrix for Nuisance Regressors

In [ ]:
# Load all regressors
mo_regressors = np.loadtxt('mo_regressors.txt')
ic_regressors = np.loadtxt('filtered_func_data.ica/melodic_mix')
rp_regressors = np.loadtxt('mc/prefiltered_func_data_mcf.par') # 6 MPs from MCFLIRT
csf_regressors = np.loadtxt('CSF.txt').reshape(-1, 1)  # reshape to column vector
wm_regressors = np.loadtxt('WM.txt').reshape(-1, 1)

# Combine all regressors into a single design matrix
regressors = np.hstack((ic_regressors, mo_regressors, rp_regressors, csf_regressors, wm_regressors))
np.savetxt('noise_regressors.txt', regressors)

# Identify noise columns: noise ICs + all motion outliers + realignment params + CSF + WM
with open('fix4melview_Standard_thr20.txt', 'r') as f:
    noise_ic_idx = f.readlines()[-1].translate(str.maketrans({"[": "", "]": "", "\n": ""})).split(", ")

# All columns after ICs are noise regressors
noise_mo_rp_csf_wm_idx = range(
    ic_regressors.shape[1] + 1,
    ic_regressors.shape[1] + mo_regressors.shape[1] + rp_regressors.shape[1] + 2 + 1
    # +2 for CSF and WM columns
)
noise_idx = ",".join([str(idx) for idx in np.hstack((noise_ic_idx, noise_mo_rp_csf_wm_idx))])

Regress out nuisance regressors

In [ ]:
!fsl_regfilt -i filtered_func_data -d noise_regressors.txt -o filtered_func_data_clean -f $noise_idx

Quality control: Nuisance Regression

In [ ]:
display(Markdown("### Raw Functional Image: Middle Volume, Brain"))

nv_FSL = AnyNiivue()
nv_FSL.load_volumes([{"path": "/content/ds005892/sub-MJF008/func/sub-MJF008_task-rest_bold.nii.gz"}])
nv_FSL

In [ ]:
display(Markdown("### Preprocessed Functional Image - Before nuisance regression"))

nv_FSL = AnyNiivue()
nv_FSL.load_volumes([{"path": "filtered_func_data.nii.gz"}])
nv_FSL


In [ ]:
display(Markdown("### Preprocessed Functional Image - After nuisance regression"))

nv_FSL = AnyNiivue()
nv_FSL.load_volumes([{"path": "filtered_func_data_clean.nii.gz"}])
nv_FSL


## Functional Connectivity Analysis

In [ ]:
import os
import matplotlib.pyplot as plt
from nilearn import datasets, maskers, connectome, plotting
os.chdir('/content/ds005892/sub-MJF008')

In [ ]:
# Load Schaefer atlas with 100 nodes
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7)
atlas_filename = atlas.maps
labels = atlas.labels[1:]  # Remove background label

In [ ]:
# Visualize the Schaefer 2018 parcellation (100 nodes) overlaid on the MNI152 T1 template

# Load MNI template as base, then overlay the atlas
display(Markdown("### Schaefer 2018 Atlas (100 nodes, Yeo 7 Networks)"))

nv_atlas = AnyNiivue()
nv_atlas.load_volumes([
    {   "path": "anat/standard_brain.nii.gz", "colormap": "gray"},
    {   "path": atlas.maps, "colormap": "random", "interpolation": "nearest",
       }
])
nv_atlas

In [ ]:
# Extract the average BOLD signal from each of the 100 regions (clean data)
masker = maskers.NiftiLabelsMasker(
    labels_img=atlas_filename,
    standardize='zscore_sample',
    memory='nilearn_cache',
    verbose=1
)

# Noisy (for comparison)
time_series_rest_noisy = masker.fit_transform('melodic_output.ica/filtered_func_data.nii.gz')
# Clean
time_series_rest_clean = masker.fit_transform('melodic_output.ica/filtered_func_data_clean.nii.gz')

In [ ]:
# Compute Pearson correlation between all pairs of regions (clean data)
correlation_measure = connectome.ConnectivityMeasure(kind='correlation', standardize='zscore_sample')

# Noisy
correlation_matrix_noisy = correlation_measure.fit_transform([time_series_rest_noisy])[0]

# Clean
correlation_matrix_clean = correlation_measure.fit_transform([time_series_rest_clean])[0]

In [ ]:
# Compute Pearson correlation between all pairs of regions (clean data)
correlation_measure = connectome.ConnectivityMeasure(kind='correlation', standardize='zscore_sample')

# Noisy
correlation_matrix_noisy = correlation_measure.fit_transform([time_series_rest_noisy])[0]

# Clean
correlation_matrix_clean = correlation_measure.fit_transform([time_series_rest_clean])[0]

Quality Control: Functional Connectivity Analysis

In [ ]:
# Plot the FC matrix - Noisy
plt.figure(figsize=(6, 4))
plt.imshow(correlation_matrix_noisy, interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('FC Matrix (Noisy)')
plt.colorbar(label='Pearson Correlation')
plt.xlabel('nodes')
plt.ylabel('nodes')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the FC matrix - Clean
plt.figure(figsize=(6, 4))
plt.imshow(correlation_matrix_clean, interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('FC Matrix (Clean)')
plt.colorbar(label='Pearson Correlation')
plt.xlabel('nodes')
plt.ylabel('nodes')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the connectivity matrix as a connectome on a brain

# Use only strong connections for clarity (threshold)
threshold = 0.6
adj_matrix = correlation_matrix_clean.copy()
adj_matrix[np.abs(adj_matrix) < threshold] = 0

# Get region coordinates from the atlas
coords = plotting.find_parcellation_cut_coords(labels_img=atlas_filename)

# Interactive 3D connectome view
view = plotting.view_connectome(
    adj_matrix,
    coords,
    edge_threshold="80%",
    node_size=10,
    colorbar=True
)
view

# Default Mode Network (DMN) Identification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

from IPython.display import display, Markdown
from ipyniivue import AnyNiivue
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi, show

import os

original_dir = '/content'
# Ensure we are in the base /content directory for consistent download location
os.chdir(original_dir)

if not os.path.exists('preprocessed_fmri.zip'):
    print('Downloading preprocessed fMRI. This should take less than 2 minutes.')
    print('Installing Gdown...')
    !pip install -q gdown
    !gdown "1_LRqGMDFfqzKwGK0C9Evvrf4zL4chr22" -O preprocessed_fmri.zip
    # Unzip directly to /content, assuming the zip contains the Neuroimaging2026/ structure
    !unzip -q -o preprocessed_fmri.zip -d /content/
    print("✓ Done")
else:
    print("✓ Already exists, skipping download.")

In [ ]:
import os

if not os.path.exists('/content/preprocessed_fmri.zip'):
    print('Downloading preprocessed fMRI. This should take less than 2 minutes.')
    print('Installing Gdown...')
    !pip install -q gdown
    !gdown "1_LRqGMDFfqzKwGK0C9Evvrf4zL4chr22" -O preprocessed_fmri.zip
    !unzip -q -o preprocessed_fmri.zip -d /content/
    print("✓ Done")
else:
    print("✓ Already exists, skipping download.")

### Get Spatial IC Maps from MELODIC output.

In [ ]:
import os

# Ensure PATTERN1 and original_dir are available
original_dir = '/content'
PATTERN1 = 'sub-MJF008'

# The melodic_IC.nii.gz is now expected in /content/Neuroimaging2026/sub-MJF008/melodic_output.ica/
INPUT_FILE=os.path.join(original_dir, 'Neuroimaging2026', PATTERN1, 'melodic_output.ica', 'melodic_IC.nii.gz')
melodic_img = nib.load(INPUT_FILE)
melodic_img_data = melodic_img.get_fdata()

# Sanity-check voxel sizes and shape
print(melodic_img.header.get_zooms())
print(melodic_img_data.shape)

### Create Yeo DMN template

In [ ]:
yeo_dataset = datasets.fetch_atlas_yeo_2011()

In [ ]:
yeo_maps = nib.load(yeo_dataset['maps'])
yeo_maps_data = yeo_maps.get_fdata()

# Create a binary mask: 1 where the value is 7, and 0 everywhere else
dmn_mask_data = (yeo_maps_data == 7).astype(np.float32)

dmn_mask_img = nib.Nifti1Image(dmn_mask_data, yeo_maps.affine, yeo_maps.header)

dmn_mask_data_3d = np.squeeze(dmn_mask_data)

dmn_mask_img = nib.Nifti1Image(dmn_mask_data_3d, yeo_maps.affine)
# Save the DMN mask to a consistent, easily accessible absolute path in /content
nib.save(dmn_mask_img, os.path.join(original_dir, 'dmn_mask_yeo.nii.gz'))

In [ ]:
plot_roi(dmn_mask_img, title="Network 7 (Default Mode Network)")
show()

### Examine space of DMN mask / atlas

In [ ]:
print(dmn_mask_img.affine)
### MNI 1 mm RAS ###

### Ensure that the Yeo Mask and Spatial IC Maps are in the same MNI space

In [ ]:
# Ensure PATTERN1 and original_dir are available
# The current working directory should be /content/ds005892/sub-MJF008/melodic_output.ica/
original_dir = '/content'
PATTERN1 = 'sub-MJF008'

# Change to the melodic output directory where these files should be created
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN1, 'melodic_output.ica'))
print(f"Current working directory for melodic_img_mni generation: {os.getcwd()}")

# Step 1:
# Use transformation matrices from fMRI preprocessing step to bring input from MELODIC (i.e., spatial IC maps) into MNI space.
# (Note: MELODIC OUTPUTS are currently in functional space)
! convert_xfm -omat func2standard.mat \
            -concat {original_dir}/ds005892/{PATTERN1}/reg/struct2standard.mat \
                    {original_dir}/ds005892/{PATTERN1}/reg/rest2struct.mat

! flirt -in {INPUT_FILE} \
      -ref {original_dir}/ds005892/{PATTERN1}/anat/standard_brain.nii.gz \
      -applyxfm \
      -init func2standard.mat \
      -out melodic_img_mni.nii.gz

print("Files in current directory after melodic_img_mni generation:")
!ls
### OUTPUT: spatial IC maps in standard MNI 2 mm ###

In [ ]:
# Step 2:
# Resample DMN mask from MNI 1 mm to MNI 2 mm
# Reference the centrally saved dmn_mask_yeo.nii.gz and the melodic_img_mni.nii.gz from CWD

# Ensure we are in the correct directory where the flirt outputs should be
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN1, 'melodic_output.ica'))
print(f"Current working directory for dmn_mask_mni generation: {os.getcwd()}")
print("Files in current directory before dmn_mask_mni generation:")
!ls

! flirt -in {original_dir}/dmn_mask_yeo.nii.gz \
      -ref melodic_img_mni.nii.gz \
      -applyxfm -usesqform \
      -interp nearestneighbour \
      -out dmn_mask_mni.nii.gz

print("Files in current directory after dmn_mask_mni generation:")
!ls
### OUTPUT: DMN Mask in standard MNI 2 mm ###

In [ ]:
import os

# Ensure we are in the correct directory where the flirt outputs should be
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN1, 'melodic_output.ica'))
print(f"Current working directory: {os.getcwd()}")
print("Files in current directory:")
!ls

# Load dmn_mask_mni.nii.gz and melodic_img_mni.nii.gz from the current directory
dmn_mask = nib.load('dmn_mask_mni.nii.gz')
melodic_img_mni = nib.load('melodic_img_mni.nii.gz')
assert np.array_equal(dmn_mask.affine, melodic_img_mni.affine)

### Binarize the IC spatial maps with cluster-defining threshold >= 2.3 *
*threshold from Woo et al., 2014 - Cluster-extent based thresholding in fMRI analyses: Pitfalls and recommendations

In [ ]:
melodic_img_mni_data = melodic_img_mni.get_fdata()
binary_ic_maps = (melodic_img_mni.get_fdata() > 2.3).astype(np.int8)

### Extract best-matching IC map with DMN mask using Dice coefficient

In [ ]:
def dice_coefficient(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    intersection = np.logical_and(a, b).sum()
    return 2 * intersection / (a.sum() + b.sum())

dice_scores = np.array([
    dice_coefficient(binary_ic_maps[..., i], dmn_mask.get_fdata())
    for i in range(binary_ic_maps.shape[-1])
])

best_ic_index = np.argmax(dice_scores)

print(f"Best IC index: {best_ic_index}, Dice: {dice_scores[best_ic_index]:.4f}")

### Obtain average Z-score within DMN mask

In [ ]:
best_ic = melodic_img_mni_data[..., best_ic_index]
mean_val = best_ic[dmn_mask.get_fdata() > 0].mean()
print(f'Average Z-score within DMN mask: {mean_val}')

### Visualize best-matching IC map.

In [ ]:
best_ic_pos = np.clip(best_ic, 0, None)
best_ic_img = nib.Nifti1Image(best_ic_pos, affine=melodic_img_mni.affine)

# Corrected path for standard_brain.nii.gz
standard_brain = nib.load(f'{original_dir}/ds005892/{PATTERN1}/anat/standard_brain.nii.gz')
title = f'Best-matching IC - Subject 8 (Dice: {dice_scores[best_ic_index]:.2f}, Mean Z-score: {mean_val:.2f})'
plotting.plot_stat_map(
    best_ic_img,
    bg_img=standard_brain,
    display_mode='ortho',
    cut_coords=(8, -66, 22),
    colorbar=True,
    cmap='hot',
    threshold=2,
    vmax=5,
    title=title
)
plotting.show()

# sub-MJF011: T1w image

In [ ]:
# Change the working directory to the structural MRI data folder of the downloaded subject
import os
os.chdir(original_dir) # Go back to the base directory first
os.chdir(os.path.join("ds005892", PATTERN2, "anat")) # Then navigate to the subject's anat folder
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Change the working directory to the structural MRI data folder of the downloaded subject
import os
os.chdir(original_dir) # Go back to the base directory first
os.chdir(os.path.join("ds005892", PATTERN2, "anat")) # Then navigate to the subject's anat folder
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Copy standard structural images from MNI to current directory
!cp $FSLDIR/data/standard/MNI152_T1_2mm.nii.gz standard.nii.gz
!cp $FSLDIR/data/standard/MNI152_T1_2mm_brain.nii.gz standard_brain.nii.gz
!cp $FSLDIR/etc/flirtsch/T1_2_MNI152_2mm.cnf .

## 1. Dataset Contents - Current Directory

In [ ]:
# List the contents of the current directory
!ls

## 2. Dataset Images

In [ ]:
# Check T1w image info
!fslinfo sub-MJF011_T1w.nii.gz

In [ ]:
# Visualize T1w image
display(Markdown("### T1w structural image"))

MJF011_T1w = AnyNiivue()
MJF011_T1w.load_volumes([{"path": "sub-MJF011_T1w.nii.gz"}])
MJF011_T1w

Images, from left to right:

Axial slices: A (anterior) on top; P (posterior) on the bottom; L (left) on the left, R (right) on the right.
Coronal slices: S (superior) on top, I (inferior) on the bottom; L (left) on the left, R (right) on the right.
Sagittal slices: S (superior) on top, I (inferior) on the bottom; P (posterior) on the left, A (anterior) on the right.
3D rendition of the brain.

## 3. Brain extraction

FSL: Brain Extraction Tool (BET)

For skull stripping, different approaches were initially considered and tested, including AFNI's `3dSkullStrip` and FreeSurfer's `SynthStrip`, alongside FSL's `BET` (Brain Extraction Tool).

Upon visual inspection of the outputs from each method, FSL's `BET` (specifically with the applied parameters) was determined to provide the most accurate and visually appealing brain extraction for the `sub-MJF011_T1w` image. It successfully removed non-brain tissue while preserving crucial brain structures, minimizing both under- and over-stripping errors. Therefore, `BET` was selected as the preferred skull-stripping tool for this project due to its superior performance on this dataset.

In [ ]:
# Run BET
!bet sub-MJF011_T1w sub-MJF011_T1w_brain -f 0.5 # define intensity threshold = 0.5

In [ ]:
# Run BET and create a binary brain mask
!bet sub-MJF011_T1w sub-MJF011_T1w_brain.nii.gz -f 0.5 -m

Result Visualization

In [ ]:
# Visualize original T1 image (before brain extraction)
display(Markdown("### T1w structural image"))

MJF011_T1w = AnyNiivue()
MJF011_T1w.load_volumes([{"path": "sub-MJF011_T1w.nii.gz"}])
MJF011_T1w

In [ ]:
# Visualize FSL's BET results
# Threshold = 0.5
display(Markdown("### FSL BET (Thr=0.5)"))

MJF011_FSL = AnyNiivue()
MJF011_FSL.load_volumes([{"path": "sub-MJF011_T1w_brain.nii.gz"}])
MJF011_FSL

In [ ]:
# Visualize FSL's BET results
# Binary brain mask
display(Markdown("### FSL BET (Thr=0.5), Brain Mask"))

MJF011_FSL = AnyNiivue()
MJF011_FSL.load_volumes([{"path": "sub-MJF011_T1w_brain_mask.nii.gz"}])
MJF011_FSL

## 4. Tissue segmentation

FAST: tissue type segmentation and bias-field correction

In [ ]:
# Run fast (with bias field correction)
!fast -b -B sub-MJF011_T1w_brain

Outputs

Bias field corrected image: sub-MJF011_T1w_brain_restore

Bias field: sub-MJF011_T1w_brain_bias

CSF image: sub-MJF011_T1w_brain_pve_0

Grey matter image: sub-MJF011_T1w_brain_pve_1

White matter image: sub-MJF011_T1w_brain_pve_2

In [ ]:
# Visualize FSL's FAST results
# Bias field
display(Markdown("### FSL FAST - Bias field"))

MJF011_T1w_FAST_bias = AnyNiivue()
MJF011_T1w_FAST_bias.load_volumes([{"path": "sub-MJF011_T1w_brain_bias.nii.gz"}])
MJF011_T1w_FAST_bias

In [ ]:
# Visualize FSL's FAST results
# Bias field corrected T1w structural image
display(Markdown("### FSL FAST - Bias field corrected T1w image"))

MJF011_T1w_FAST_restore = AnyNiivue()
MJF011_T1w_FAST_restore.load_volumes([{"path": "sub-MJF011_T1w_brain_restore.nii.gz"}])
MJF011_T1w_FAST_restore

In [ ]:
# Visualize FSL's FAST results
# CSF
display(Markdown("### FSL FAST - CSF"))

MJF011_T1w_FAST_CSF = AnyNiivue()
MJF011_T1w_FAST_CSF.load_volumes([{"path": "sub-MJF011_T1w_brain_pve_0.nii.gz"}])
MJF011_T1w_FAST_CSF

In [ ]:
# Visualize FSL's FAST results
# Grey matter
display(Markdown("### FSL FAST - Grey matter"))

MJF011_T1w_FAST_GM = AnyNiivue()
MJF011_T1w_FAST_GM.load_volumes([{"path": "sub-MJF011_T1w_brain_pve_1.nii.gz"}])
MJF011_T1w_FAST_GM

In [ ]:
# Visualize FSL's FAST results
# White Matter
display(Markdown("### FSL FAST - White matter"))

MJF011_T1w_FAST_WM = AnyNiivue()
MJF011_T1w_FAST_WM.load_volumes([{"path": "sub-MJF011_T1w_brain_pve_2.nii.gz"}])
MJF011_T1w_FAST_WM

In [ ]:
# Visualize FSL's FAST results
# CSF, grey matter, white matter (overlayed)
display(Markdown("### FSL FAST - CSF (red), GM (blue), WM (white)"))

MJF011_T1w_FAST_all = AnyNiivue()
volumes = [{"path": "sub-MJF011_T1w_brain_pve_2.nii.gz", "colormap": "white"},
           {"path": "sub-MJF011_T1w_brain_pve_0.nii.gz", "colormap": "red"},
           {"path": "sub-MJF011_T1w_brain_pve_1.nii.gz", "colormap": "blue"}]
MJF011_T1w_FAST_all.load_volumes(volumes)
MJF011_T1w_FAST_all

After segmentation with FAST, the segmented grey matter, white matter, and CSF masks were used to estimate the total intracranial volume (ICV). This value was later used to normalize the extracted subcortical volumes, enabling more reliable anatomical comparisons.

In [ ]:
# Intracranial Volume

# Tissues volume
# CSF volume
CSF = !fslstats sub-MJF011_T1w_brain_pve_0 -V
CSF_vol = float(CSF[0].split()[1])

# GM volume
GM = !fslstats sub-MJF011_T1w_brain_pve_1 -V
GM_vol = float(GM[0].split()[1])

# WM volume
WM = !fslstats sub-MJF011_T1w_brain_pve_2 -V
WM_vol = float(WM[0].split()[1])

# ICV (Total volume)
ICV = CSF_vol + GM_vol + WM_vol

print(f"CSF: {CSF_vol} mm³")
print(f"GM: {GM_vol} mm³")
print(f"WM: {WM_vol} mm³")
print(f"ICV: {ICV} mm³")

FIRST: segmentation of sub-cortical structures

In [ ]:
# Perform segmentation of sub-cortical structures
!run_first_all -i sub-MJF011_T1w_brain -b -o MJF011_T1w_brain_segmented

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation"))

MJF011_T1w_FSL = AnyNiivue()
MJF011_T1w_FSL.load_volumes([{"path": "MJF011_T1w_brain_segmented_all_fast_firstseg.nii.gz"}])
MJF011_T1w_FSL

## 5. Volume Extraction

Volume extraction of 3 subcortical structures of interest: putamen, hippocampus, amygdala.

Since the brains of the two subjects have different dimensions, we had to perform the normalization of the volumes in order to accurately compare them, using the intracranial value we calculated before.

1. Putamen

In [ ]:
# Volume extraction of Putamen

# Left Putamen (label 12)
L_Puta = !fslstats MJF011_T1w_brain_segmented_all_fast_firstseg -l 11.5 -u 12.5 -V
L_Puta_vol = float(L_Puta[0].split()[1])

# Right Putamen (label 51)
R_Puta = !fslstats MJF011_T1w_brain_segmented_all_fast_firstseg -l 50.5 -u 51.5 -V
R_Puta_vol = float(R_Puta[0].split()[1])

# Putamen total volume
Puta = L_Puta_vol + R_Puta_vol

print(f"Left: {L_Puta_vol} mm³")
print(f"Right: {R_Puta_vol} mm³")
print(f"Puta: {Puta} mm³")

In [ ]:
# Putamen volume Normalization
Puta_norm = Puta / ICV

print(f"Puta_norm: {Puta_norm}")

Visualization

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation of Putamen (Putamen - Red and T1w - Blue)"))

# Extract Left Putamen mask (label 12)
!fslmaths MJF011_T1w_brain_segmented_all_fast_firstseg -thr 12 -uthr 12 -bin MJF011_putamen_L_Puta.nii.gz

# Extract Right Putamen mask (label 51)
!fslmaths MJF011_T1w_brain_segmented_all_fast_firstseg -thr 51 -uthr 51 -bin MJF011_putamen_R_Puta.nii.gz

MJF008_T1w_FSL = AnyNiivue()
volumes = [{"path": "MJF011_putamen_L_Puta.nii.gz", "colormap":"red"},
           {"path": "MJF011_putamen_R_Puta.nii.gz", "colormap": "red"},
           {"path": "sub-MJF011_T1w_brain.nii.gz", "colormap": "blue","opacity": 0.4}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

2. Hippocampus

In [ ]:
# Volume extraction of Hippocampus

# Left Hippocampus (label 17)
L_Hipp = !fslstats MJF011_T1w_brain_segmented_all_fast_firstseg -l 16.5 -u 17.5 -V
L_Hipp_vol = float(L_Hipp[0].split()[1])

# Right Hippocampus (label 53)
R_Hipp = !fslstats MJF011_T1w_brain_segmented_all_fast_firstseg -l 52.5 -u 53.5 -V
R_Hipp_vol = float(R_Hipp[0].split()[1])

# Hippocampus total volume
Hipp = L_Hipp_vol + R_Hipp_vol

print(f"Left: {L_Hipp_vol} mm³")
print(f"Right: {R_Hipp_vol} mm³")
print(f"Hipp: {Hipp} mm³")

In [ ]:
# Hippocampus volume Normalization
Hipp_norm = Hipp / ICV

print(f"Hipp_norm: {Hipp_norm}")

Visualization

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation of Hippocampus (Hippocampus - Red and T1w - Blue)"))

# Extract Left Hippocampus mask (label 17)
!fslmaths MJF011_T1w_brain_segmented_all_fast_firstseg -thr 17 -uthr 17 -bin MJF011_hippocampus_L_Hipp.nii.gz

# Extract Right Hippocampus mask (label 53)
!fslmaths MJF011_T1w_brain_segmented_all_fast_firstseg -thr 53 -uthr 53 -bin MJF011_hippocampus_R_Hipp.nii.gz

MJF008_T1w_FSL = AnyNiivue()
volumes = [{"path": "MJF011_hippocampus_L_Hipp.nii.gz", "colormap":"red"},
           {"path": "MJF011_hippocampus_R_Hipp.nii.gz", "colormap": "red"},
           {"path": "sub-MJF011_T1w_brain.nii.gz", "colormap": "blue","opacity": 0.4}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

3. Amygdala

In [ ]:
# Volume extraction of Amygdala

# Left Amygdala (label 18)
L_Amyg = !fslstats MJF011_T1w_brain_segmented_all_fast_firstseg -l 17.5 -u 18.5 -V
L_Amyg_vol = float(L_Amyg[0].split()[1])

# Right Amygdala (label 54)
R_Amyg = !fslstats MJF011_T1w_brain_segmented_all_fast_firstseg -l 53.5 -u 54.5 -V
R_Amyg_vol = float(R_Amyg[0].split()[1])

# Amygdala total volume
Amyg = L_Amyg_vol + R_Amyg_vol

print(f"Left: {L_Amyg_vol} mm³")
print(f"Right: {R_Amyg_vol} mm³")
print(f"Amyg: {Amyg} mm³")

In [ ]:
# Amygdala volume Normalization
Amyg_norm = Amyg / ICV

print(f"Amyg_norm: {Amyg_norm}")

Visualization

In [ ]:
# Visualize FSL results
# FIRST
display(Markdown("### FSL FIRST - Segmentation of Amygdala (Putamen - Red and Brain - Blue)"))

# Extract Left Amygdala mask (label 18)
!fslmaths MJF011_T1w_brain_segmented_all_fast_firstseg -thr 18 -uthr 18 -bin MJF011_amygdala_L_Amyg.nii.gz

# Extract Right Amygdala mask (label 54)
!fslmaths MJF011_T1w_brain_segmented_all_fast_firstseg -thr 54 -uthr 54 -bin MJF011_amygdala_R_Amyg.nii.gz

MJF008_T1w_FSL = AnyNiivue()
volumes = [{"path": "MJF011_amygdala_L_Amyg.nii.gz", "colormap":"red"},
           {"path": "MJF011_amygdala_R_Amyg.nii.gz", "colormap": "red"},
           {"path": "sub-MJF011_T1w_brain.nii.gz", "colormap": "blue","opacity": 0.4}]
MJF008_T1w_FSL.load_volumes(volumes)
MJF008_T1w_FSL

## 6. Registration

FNIRT: Between-subject registration

To spatially normalize the subject’s T1-weighted image to the MNI standard space (MNI152_T1), a two-step registration approach was adopted.

An initial affine linear registration was first performed using FLIRT to estimate the global transformation between the subject and template images.

This was followed by non-linear registration with FNIRT, which allowed local anatomical differences to be modeled, resulting in a more precise alignment to the standard space.

In [ ]:
# Visualize results (before registration)
display(Markdown("### Standard (blue) and T1w (red) - before registration"))

MJF011_T1w_FSL = AnyNiivue()
volumes =  [{"path": "standard_brain.nii.gz", "colormap": "blue"},
           {"path": "sub-MJF011_T1w_brain.nii.gz", "colormap": "red"}]
MJF011_T1w_FSL.load_volumes(volumes)
MJF011_T1w_FSL

In [ ]:
# It's common to use FLIRT to obtain an initial approximation of the transformation, which we can later input in FNIRT
# Use an affine linear transformation with 12 DOF
!flirt -in sub-MJF011_T1w_brain.nii.gz -ref standard_brain -dof 12 -out sub-MJF011_T1wtoMNIlin -omat sub-MJF011_T1wtoMNIlin.mat

# Provide the affine transformation matrix estimated in the step above to initialize the non-linear registration
!fnirt --in=sub-MJF011_T1w_brain.nii.gz --aff=sub-MJF011_T1wtoMNIlin.mat --config=T1_2_MNI152_2mm.cnf --iout=sub-MJF011_T1wtoMNInonlin --cout=sub-MJF008_T1wtoMNI_coef --fout=sub-MJF008_T1wtoMNI_warp

In [ ]:
# Visualize results (after registration)
display(Markdown("### Standard MNI (blue) and T1w (red) - after non linear registration"))

MJF011_T1w_FNIRT = AnyNiivue()
volumes =  [{"path": "sub-MJF011_T1wtoMNIlin.nii.gz", "colormap": "blue"},
           {"path": "standard_brain.nii.gz", "colormap": "red"}]
MJF011_T1w_FNIRT.load_volumes(volumes)
MJF011_T1w_FNIRT

In [ ]:
# Visualize results (after registration)
display(Markdown("### Standard MNI (blue) and T1w (red) - after non linear registration"))

MJF011_T1w_FNIRT = AnyNiivue()
volumes =  [{"path": "sub-MJF011_T1wtoMNInonlin.nii.gz", "colormap": "blue"},
           {"path": "standard_brain.nii.gz", "colormap": "red"}]
MJF011_T1w_FNIRT.load_volumes(volumes)
MJF011_T1w_FNIRT

# sub-MJF011: fMRI image

## 1. Change Directory

In [ ]:
# Change the working directory to the fieldmap data folder of the downloaded subject
import os

# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_fmap_dir = os.path.join("ds005892", PATTERN2, "fmap")

# Change to the target directory
os.chdir(target_fmap_dir)

print(f"Current directory: {os.getcwd()}")

Dataset Contents - Current Directory

In [ ]:
# List the contents of the current directory
!ls

## 2. Preprocessing of fMRI data

 ### Prepare Fieldmap

In [ ]:
# Obtain the delta echo time and the dwell time from the magnitude1 .json file
!cat sub-MJF011_magnitude1.json

In [ ]:
# Assign delta echo time and dwell time variables
delta_echo_time = 2.46 # ms
dwell_time = 0.008 # s

In [ ]:
# Skull-strip the fieldmap magnitude image (magnitude2 is superfluous and therefore ignored)
!bet sub-MJF011_magnitude1 sub-MJF011_magnitude1_brain

# Erode (shaving off voxels from the edges) the skull-stripped fieldmap magnitude image
!fslmaths sub-MJF011_magnitude1_brain -ero sub-MJF011_magnitude1_brain_eroded

In [ ]:
# Visualize the fieldmap magnitude images - before erosion
display(Markdown("### Fieldmap magnitude image - before erosion "))

nv_fmap_mag = AnyNiivue()
nv_fmap_mag.load_volumes([{"path": "sub-MJF011_magnitude1_brain.nii.gz"}])
nv_fmap_mag


In [ ]:
# Visualize the fieldmap magnitude images - after erosion
display(Markdown("### Fieldmap magnitude image - after erosion "))

nv_fmap_mag_ero = AnyNiivue()
nv_fmap_mag_ero.load_volumes([{"path": "sub-MJF011_magnitude1_brain_eroded.nii.gz"}])
nv_fmap_mag_ero

In [ ]:
# Visualize the original fieldmap phase difference image
display(Markdown("### Fieldmap phase difference image "))

nv_fmap_phase = AnyNiivue()
nv_fmap_phase.load_volumes([{"path": "sub-MJF011_phasediff.nii.gz"}])
nv_fmap_phase

In [ ]:
# Fieldmap preparation
# Convert the raw phase difference image into a fieldmap in radians/second,
# using the delta_echo_time (TE2-TE1) to calibrate the phase to frequency conversion

# fsl_prepare_fieldmap <scanner> <phase_image> <magnitude_image> <out_image> <deltaTE (in ms)>
!fsl_prepare_fieldmap SIEMENS sub-MJF011_phasediff sub-MJF011_magnitude1_brain_eroded sub-MJF011_rads $delta_echo_time

In [ ]:
# Visualize preprocessed fieldmap
display(Markdown("### Preprocessed fieldmap "))

nv_fmap = AnyNiivue()
nv_fmap.load_volumes([{"path": "sub-MJF011_rads.nii.gz"}])
nv_fmap

### Distortion Correction and Registration

Change Directory

In [ ]:
# Change the working directory to the rs-fMRI data folder of the downloaded subject
import os

# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_func_dir = os.path.join("ds005892", PATTERN2, "func")

# Change to the target directory
os.chdir(target_func_dir)

print(f"Current directory: {os.getcwd()}")

In [ ]:
# Visualize the functional image (one volume)
display(Markdown("### Functional Image (One Volume)"))

MJF011_fmri = AnyNiivue()
MJF011_fmri.load_volumes([{"path": "sub-MJF011_task-rest_bold.nii.gz"}])
MJF011_fmri

Dataset Contents - Current Directory

In [ ]:
# List the contents of the current directory
!ls

In [ ]:
# Obtain the phase encoding direction for epi_reg
# -j is anterior to posterior (-y)

!cat sub-MJF011_task-rest_bold.json

Dataset Images

In [ ]:
echo_time_func = 0.029
dwell_time_func = 0.000590001

In [ ]:
# Check rs-fMRI image info, especially total number of volumes (dim4)
!fslinfo sub-MJF011_task-rest_bold.nii.gz

Obtain a 3D image from the 4D functional data (for registration)

In [ ]:
# Use the middle volume as a representative single image for registration
tstart = 100 # Middle volume
tsize = 1

# Extract the middle volume
!fslroi sub-MJF011_task-rest_bold  sub-MJF011_task-rest_bold_midvol $tstart $tsize

# Skull-strip the extracted middle volume
!bet sub-MJF011_task-rest_bold_midvol sub-MJF011_task-rest_bold_midvol_brain

In [ ]:
# Visualize the functional image (middle volume - brain)
display(Markdown("### Functional Image (Middle Volume - Brain)"))

MJF011_fmri_midvol_brain = AnyNiivue()
MJF011_fmri_midvol_brain.load_volumes([{"path": "sub-MJF011_task-rest_bold_midvol_brain.nii.gz"}])
MJF011_fmri_midvol_brain

Distortion correction and registration to structural space

In [ ]:
# Change the working directory to the downloaded subject
import os

# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_dir = os.path.join("ds005892", PATTERN2)

# Change to the target directory
os.chdir(target_dir)

print(f"Current directory: {os.getcwd()}")

In [ ]:
# epi_reg inputs (use file names):
# --epi: Middle fMRI volume (brain extracted)
# --t1: Wholehead T1 image
# --t1brain: Brain extracted T1 image (not mask)
# --out: Output name
# --fmap: Fieldmap image (in rad/s)
# --fmapmag: Fieldmap magnitude image - wholehead extracted
# --fmapmagbrain: Fieldmap magnitude image - brain extracted
# --echospacing: Dwell time (s)
# --pedir: Phase encoding direction

# Main outputs:
# rest2struct.nii.gz: EPI (distortion-corrected) in T1 space
# rest2struct.mat: EPI → T1 transform
# rest2struct_warp.nii.gz: Nonlinear distortion warp (fmap)

# Create a folder for registration files
!mkdir -p reg

!epi_reg --epi=func/sub-MJF011_task-rest_bold_midvol_brain --t1=anat/sub-MJF011_T1w --t1brain=anat/sub-MJF011_T1w_brain --out=reg/rest2struct --fmap=fmap/sub-MJF011_rads --fmapmag=fmap/sub-MJF011_magnitude1 --fmapmagbrain=fmap/sub-MJF011_magnitude1_brain_eroded --echospacing=$dwell_time_func --pedir=-y --noclean

In [ ]:
# Obtain transformation matrix from strucutral space to undistorted functional space
# !convert_xfm -options <output> <input>
# struct2rest.mat: T1 → EPI transform (linear)
!convert_xfm -inverse -omat reg/struct2rest.mat reg/rest2struct.mat

# Unwarp example_func in structural space (rest2struct) to obtain undistorted functional image (rest_bold_midvol_brain_undistorted)
# Takes the corrected functional image that epi_reg placed in structural space (rest2struct) and brings it back into functional space using the inverse transform
!applywarp -i func/sub-MJF011_task-rest_bold_midvol_brain -r func/sub-MJF011_task-rest_bold_midvol_brain --premat=reg/rest2struct.mat --postmat=reg/struct2rest.mat -o func/sub-MJF011_task-rest_bold_midvol_brain_undistorted


Quality control of distortion correction and registration

In [ ]:
# Visualize the results WITHOUT distortion correction
display(Markdown("### B0 unwarping of functional volume: before"))

nv_distorted = AnyNiivue()
nv_distorted.load_volumes([{"path": "func/sub-MJF011_task-rest_bold_midvol_brain.nii.gz"}])
nv_distorted


In [ ]:
# Visualize the results WITH distortion correction
display(Markdown("### B0 unwarping of functional volume: after"))

nv_undistorted = AnyNiivue()
nv_undistorted.load_volumes([{"path": "func/sub-MJF011_task-rest_bold_midvol_brain_undistorted.nii.gz"}])
nv_undistorted

In [ ]:
# For registration comparison: register the distorted to structural image
!flirt -ref anat/sub-MJF011_T1w -in func/sub-MJF011_task-rest_bold_midvol_brain -out func/sub-MJF011_task-rest_bold_midvol_brain_distorted -applyxfm -init reg/rest2struct.mat -interp trilinear


In [ ]:
# Visualize the results of registration (EPI -> T1 -> EPI): WITHOUT DISTORTION CORRECTION
display(Markdown("### Registration of functional (red) to structural (blue) space: without distortion correction"))

nv_reg_func2highres_distorted = AnyNiivue()
volumes =  [{"path": "anat/sub-MJF011_T1w.nii.gz", "colormap": "blue"},
           {"path": "func/sub-MJF011_task-rest_bold_midvol_brain_distorted.nii.gz", "colormap": "red"}]
nv_reg_func2highres_distorted.load_volumes(volumes)
nv_reg_func2highres_distorted


In [ ]:
# Visualize the results of registration (EPI -> T1 -> EPI): WITH DISTORTION CORRECTION
display(Markdown("### Registration of functional (red) to structural (blue) space: WITH distortion correction"))

nv_reg_func2highres_undistorted = AnyNiivue()
volumes =  [{"path": "anat/sub-MJF011_T1w.nii.gz", "colormap": "blue"},
           {"path": "func/sub-MJF011_task-rest_bold_midvol_brain_undistorted.nii.gz", "colormap": "red"}]
nv_reg_func2highres_undistorted.load_volumes(volumes)
nv_reg_func2highres_undistorted

In [ ]:
# Visualize the results of registration (EPI -> T1): WITH DISTORTION CORRECTION
display(Markdown("### Registration of functional (red) to structural (blue) space: with distortion correction"))

nv_reg_func2highres_pos = AnyNiivue()
volumes =  [{"path": "anat/sub-MJF011_T1w.nii.gz", "colormap": "blue"},
           {"path": "reg/rest2struct.nii.gz", "colormap": "red"}]
nv_reg_func2highres_pos.load_volumes(volumes)
nv_reg_func2highres_pos

Perform registration to standard (MNI) space

In [ ]:
# Register highres space to standard space
!flirt -in anat/sub-MJF011_T1w_brain -ref anat/standard_brain -dof 12 -out reg/struct2standard -omat reg/struct2standard.mat

# Register functional space (already in structural space) to standard space
!flirt -in reg/rest2struct.nii.gz -ref anat/standard_brain -applyxfm -init reg/struct2standard.mat -out reg/func2standard


In [ ]:
# Visualize the results of registration: functional to standard space
# Before
display(Markdown("### Registration of functional (red) to standard (blue) space: before"))

nv_reg_func2std_pre = AnyNiivue()
volumes =  [{"path": "anat/standard_brain.nii.gz", "colormap": "blue"},
           {"path": "func/sub-MJF011_task-rest_bold_midvol_brain_undistorted.nii.gz", "colormap": "red"}]
nv_reg_func2std_pre.load_volumes(volumes)
nv_reg_func2std_pre

In [ ]:
# Visualize the results of registration: functional to standard space
# After
display(Markdown("### Registration of functional (red) to standard (blue) space: after"))

nv_reg_func2std_pos = AnyNiivue()
volumes =  [{"path": "anat/standard_brain.nii.gz", "colormap": "blue"},
           {"path": "reg/func2standard.nii.gz", "colormap": "red"}]
nv_reg_func2std_pos.load_volumes(volumes)
nv_reg_func2std_pos

### Motion Correction

Using MCFLIRT, which loads the entire time-series and defaults to the middle (100) volume as initial template, it finds the transformation to the adjacent volume and uses this as as estimate for the next transformation.

Output: estimation of 6 motion realignment parameters: 3 translations and 3 rotations

In [ ]:
# Use MCFLIRT to estimate 6 realignment parameters (RP) and correct distorted functional image for motion artefacts
# Removed reffile since refvol defaults to middle volume
# Out: sub-MJF008_task-rest_bold_mcf

import os

# Change directory to func
# Ensure we are starting from the base directory where ds005892 was installed
os.chdir(original_dir)

# Construct the full path to the target func directory
target_func_dir = os.path.join("ds005892", PATTERN2, "func")

# Change to the target directory
os.chdir(target_func_dir)

# Create folder for motion correction files
!mkdir -p mc

# Run motion correction
!mcflirt -in sub-MJF011_task-rest_bold -mats -plots -rmsrel -rmsabs -spline_final

In [ ]:
# MCFLIRT estimated rotations in radians (three)
!fsl_tsplot -i sub-MJF011_task-rest_bold_mcf.par -t 'MCFLIRT-estimated rotations (radians)' -u 1 --start=1 --finish=3 -a x,y,z -w 640 -h 144 -o rot.png

# MCFLIRT estimated translations in mm (three)
!fsl_tsplot -i sub-MJF011_task-rest_bold_mcf.par -t 'MCFLIRT-estimated translations (mm)' -u 1 --start=4 --finish=6 -a x,y,z -w 640 -h 144 -o trans.png

# MCFLIRT estimated mean displacement in mm
!fsl_tsplot -i sub-MJF011_task-rest_bold_mcf_abs.rms,sub-MJF011_task-rest_bold_mcf_rel.rms -t 'MCFLIRT-estimated mean displacement (mm)' -u 1 -w 640 -h 144 -a absolute,relative -o disp.png

Quality Control: Motion correction

In [ ]:
from IPython.display import Image
Image(filename='rot.png')

# Relatively small values, indicates gradual drift

In [ ]:
Image(filename='trans.png')

# Relatively small values

In [ ]:
Image(filename='disp.png')

# Typical

In [ ]:
import os

# Change directory to func where mcflirt outputs are
os.chdir(os.path.join(original_dir, "ds005892", PATTERN2, "func"))

# Create mc subdirectory inside func if it doesn't exist
!mkdir -p mc

# Move mcflirt-generated files and directory into mc/
# The .mat directory is sub-MJF011_task-rest_bold_mcf.mat
!mv sub-MJF011_task-rest_bold_mcf.mat mc/
!mv sub-MJF011_task-rest_bold_mcf.par mc/
!mv sub-MJF011_task-rest_bold_mcf_abs.rms mc/
!mv sub-MJF011_task-rest_bold_mcf_abs_mean.rms mc/
!mv sub-MJF011_task-rest_bold_mcf_rel.rms mc/
!mv sub-MJF011_task-rest_bold_mcf_rel_mean.rms mc/
!mv rot.png mc/
!mv trans.png mc/
!mv disp.png mc/

# Concatenate all 6 estimated realignment parameters in epi_mcf.cat
# Now within func, so paths are relative to func
!cat mc/sub-MJF011_task-rest_bold_mcf.mat/MAT* > mc/sub-MJF011_task-rest_bold_mcf.cat

In [ ]:
# Generate carpet plot to visualize motion artifacts
from nilearn import plotting
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
plotting.plot_carpet(
    'func/sub-MJF011_task-rest_bold.nii.gz',
    figure=fig,
    axes=ax,
    title="Carpet Plot",
    standardize='zscore_sample',
    t_r=None
)

plt.show()

### Unwarp and skull-strip functional data

In [ ]:
import os

# Ensure we are in the correct subject directory (/content/ds005892/sub-MJF011)
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN2))
current_subject_dir = os.getcwd()

print(f"Current working directory: {current_subject_dir}")
print("Verifying existence of input files for applywarp:")

# Define absolute paths for all files
func_path = os.path.join(current_subject_dir, 'func')
reg_path = os.path.join(current_subject_dir, 'reg')
mc_path = os.path.join(func_path, 'mc')

input_image_path = os.path.join(func_path, 'sub-MJF011_task-rest_bold_mcf.nii.gz')
ref_image_path = os.path.join(func_path, 'sub-MJF011_task-rest_bold_midvol_brain_undistorted.nii.gz')
output_image_path = os.path.join(func_path, 'sub-MJF011_task-rest_bold_unwarp.nii.gz')
premat_path = os.path.join(mc_path, 'sub-MJF011_task-rest_bold_mcf.cat')
warp_path = os.path.join(reg_path, 'rest2struct_warp.nii.gz')
postmat_path = os.path.join(reg_path, 'struct2rest.mat')

print(f"Input image ({input_image_path}) exists: {os.path.exists(input_image_path)}")
print(f"Reference image ({ref_image_path}) exists: {os.path.exists(ref_image_path)}")
print(f"Pre-matrix file ({premat_path}) exists: {os.path.exists(premat_path)}")
print(f"Warp file ({warp_path}) exists: {os.path.exists(warp_path)}")
print(f"Post-matrix file ({postmat_path}) exists: {os.path.exists(postmat_path)}")

!applywarp -i {input_image_path} -r {ref_image_path} -o {output_image_path} --premat={premat_path} --warp={warp_path} --postmat={postmat_path} --rel --interp=spline --paddingsize=1

In [ ]:
# Skull-strip 4D functional data
# BET expects a 3D image so we need to compute the temporal mean, run BET on that single average volume to create a brain mask, then apply that mask to the full 4D data

# Change directory to subject/func
os.chdir('/content/ds005892/sub-MJF011/func')

# Extract temporal average of 4D data
!fslmaths sub-MJF011_task-rest_bold_unwarp -Tmean sub-MJF011_task-rest_bold_unwarp_mean

# Perform brain extraction on averaged 4D (now 3D) data and create brain mask (mask)
!bet2 sub-MJF011_task-rest_bold_unwarp_mean mask -f 0.3 -n -m; immv mask_mask mask

# Create skull-stripped time-series (epi_bet) with mask
!fslmaths sub-MJF011_task-rest_bold_unwarp -mas mask sub-MJF011_task-rest_bold_unwarp_brain

Quality control: Unwarping and skull-stripping of the entire functional volume

In [ ]:
# Visualize the results of B0 unwarping and brain extraction on a volume of the functional data
# Before
from IPython.display import display, Markdown
display(Markdown("### Unwarping and brain extraction: before"))

nv_unwarp_bet_pre = AnyNiivue()
nv_unwarp_bet_pre.load_volumes([{"path": "sub-MJF011_task-rest_bold_mcf.nii.gz"}])
nv_unwarp_bet_pre

In [ ]:
# Visualize the results of B0 unwarping and brain extraction on a volume of the functional data
# After
display(Markdown("### Unwarping and brain extraction: after"))

nv_unwarp_bet_pos = AnyNiivue()
nv_unwarp_bet_pos.load_volumes([{"path": "sub-MJF011_task-rest_bold_unwarp_brain.nii.gz"}])
nv_unwarp_bet_pos

### High-pass Temporal Filtering

In [ ]:
# Check TR
!fslinfo sub-MJF011_task-rest_bold_unwarp_brain

In [ ]:
# Define hyperparameters
TR = 2 # s
hp_freq = 0.01 # Hz
hp_sigma = 1 / (2 * TR * hp_freq)

# Mean of brain-extracted data
!fslmaths sub-MJF011_task-rest_bold_unwarp_brain -Tmean sub-MJF011_task-rest_bold_unwarp_brain_mean

# Apply high-pass filter
!fslmaths sub-MJF011_task-rest_bold_unwarp_brain -bptf {hp_sigma} -1 -add sub-MJF011_task-rest_bold_unwarp_brain_mean sub-MJF011_task-rest_bold_unwarp_brain_hpf

Quality control: High-pass filtering

In [ ]:
# Visualize the results of high pass filtering on a volume of the functional data
# Before
display(Markdown("### High pass temporal filtering of functional data: before"))

nv_hpf_pre = AnyNiivue()
nv_hpf_pre.load_volumes([{"path": "sub-MJF011_task-rest_bold_unwarp_brain.nii.gz"}])
nv_hpf_pre

In [ ]:
# Visualize the results of high pass filtering on a volume of the functional data
# AFTER
display(Markdown("### High pass temporal filtering of functional data: after"))

nv_hpf_post = AnyNiivue()
nv_hpf_post.load_volumes([{"path": "sub-MJF011_task-rest_bold_unwarp_brain_hpf.nii.gz"}])
nv_hpf_post

### Spatially smooth functional data

In [ ]:
# Check voxel size - anisotropic
!fslinfo sub-MJF011_task-rest_bold_unwarp_brain_hpf

In [ ]:
# Obtain geometric mean of voxel size (as a compromise) and use it as basis for fwhm
geom_mean_voxel = np.cbrt(3.125 * 3.125 * 3.5)

# Define hyperparameters
fwhm = 1.5 * geom_mean_voxel
sigma = fwhm / 2.355

In [ ]:
# Calculate the median brain intensity (50th percentile) within the brain mask
# Use NON-HPF data for brightness threshold (stable intensities)
median=!fslstats sub-MJF011_task-rest_bold_unwarp_brain -k mask -p 50

# Set brightness threshold at 75% of median — voxels further apart than this will not be averaged together
bright_thr = float(median[0]) * 0.75

# Same calculation for the reference image (temporal mean)
# Use NON-HPF mean as USAN reference (standard practice)
median_usan=!fslstats sub-MJF011_task-rest_bold_unwarp_brain_mean -k mask -p 50
bright_thr_usan = float(median_usan[0]) * 0.75

# Run SUSAN smoothing: edge-preserving Gaussian blur using mean_func as reference# Run SUSAN smoothing: edge-preserving Gaussian blur using mean_func as reference
# Use HPF data
!susan sub-MJF011_task-rest_bold_unwarp_brain_hpf $bright_thr $sigma 3 1 1 sub-MJF011_task-rest_bold_unwarp_brain_mean $bright_thr_usan sub-MJF011_task-rest_bold_unwarp_brain_hpf_smooth

# Reapply brain mask to remove any signal smoothed outside the brain boundary
!fslmaths sub-MJF011_task-rest_bold_unwarp_brain_hpf_smooth -mas mask sub-MJF011_task-rest_bold_unwarp_brain_hpf_smooth.nii.gz

In [ ]:
# Visualize the results of spatial smoothing on a volume of the functional data
# Before
display(Markdown("### Spatial smoothing of functional data: before"))

nv_smoothing_bet_pre = AnyNiivue()
nv_smoothing_bet_pre.load_volumes([{"path": "sub-MJF011_task-rest_bold_unwarp_brain_hpf.nii.gz"}])
nv_smoothing_bet_pre

In [ ]:
# Visualize the results of spatial smoothing on a volume of the functional data
# After
display(Markdown("### Spatial smoothing of functional data: after"))

nv_smoothing_pos = AnyNiivue()
nv_smoothing_pos.load_volumes([{"path": "sub-MJF011_task-rest_bold_unwarp_brain_hpf_smooth.nii.gz"}])
nv_smoothing_pos

### Perform ICA

In [ ]:
# Input: 4D images that have been:
# 1. Distortion corrected
# 2. Registered to structural space
# 3. Motion corrected
# 4. Skull-stripped
# 5. High-pass temporal filtered
# 6. Spatially-smoothed

# Output: melodic_IC.nii.gz
# 4D image where each volume = one independent component (IC)

# Move to subject folder
os.chdir('/content/ds005892/sub-MJF011')

# Run MELODIC
!melodic -i func/sub-MJF011_task-rest_bold_unwarp_brain_hpf_smooth -o melodic_output.ica --nobet --tr=$TR --report

### Classify and remove noise components from single-subject ICA

In [ ]:
# Create subdirectories inside melodic output to mimic FEAT structure
!mkdir -p melodic_output.ica/filtered_func_data.ica
!mkdir -p melodic_output.ica/reg
!mkdir -p melodic_output.ica/mc

# Copy ICA files into the subdirectory
!cp melodic_output.ica/melodic_* melodic_output.ica/filtered_func_data.ica/

# Copy preprocessed 4D data into the subdirectory. Rename the 4D file as filtered_func_data
!cp func/sub-MJF011_task-rest_bold_unwarp_brain_hpf_smooth.nii.gz melodic_output.ica/filtered_func_data.nii.gz

# Copy necessary supporting files and rename based of MELODIC/FEAT convention (FIX/UserGuide)
!cp anat/sub-MJF011_T1w_brain.nii.gz melodic_output.ica/reg/highres.nii.gz  # brain-extracted structural
!cp reg/struct2rest.mat melodic_output.ica/reg/highres2example_func.mat     # FLIRT transform from structural to functional space
!cp func/mc/sub-MJF011_task-rest_bold_mcf.par melodic_output.ica/mc/prefiltered_func_data_mcf.par # motion parameters created by mcflirt
!cp func/sub-MJF011_task-rest_bold_midvol_brain_undistorted.nii.gz melodic_output.ica/reg/example_func.nii.gz # Middle volume of 4D data as example

# Remain commented these lines
# !cp reg/rest2struct.mat melodic_output.ica/reg/example_func2highres.mat
# !cp melodic_output.ica/mask.nii.gz melodic_output.ica/filtered_func_data.ica/mask.nii.gz
# !cp melodic_output.ica/mean.nii.gz melodic_output.ica/filtered_func_data.ica/mean_func.nii.gz   # 3D image (temporal mean of 4D)
# !cp /content/ds005892/sub-MJF011/anat/standard.nii.gz melodic_output.ica/reg
# !cp sub-MJF011_task-rest_bold_midvol_brain_undistorted.nii.gz melodic_output.ica/reg/example_func.nii.gz

# Feature extraction (required for classification)
# Output: melodic_output.ica/fix
!fix -f melodic_output.ica

# Classification with pretrained classifier (Standard) with a usual threshold
# P(noise) > 20 - remove component
# Output: fix4melview_Standard_thr20.txt
!fix -c melodic_output.ica Standard 20

# Change directory to melodic output and inspect the classification results
os.chdir('/content/ds005892/sub-MJF011/melodic_output.ica')
!cat fix4melview_Standard_thr20.txt

### Nuisance Regression

Motion Outliers

In [ ]:
# Use the skull-stripped time-series as input to identify motion outliers
# Use RMS intensity difference to reference volume as metric
!fsl_motion_outliers -i filtered_func_data -o mo_regressors.txt --refrms --nomoco -m mask -s mc/refrms.txt -p mc/refrms

# Rows (volumes), columns (type of MO), 1 = presence of MO

In [ ]:
# Check the contents of the confound matrix created
# Each column represents one motion outlier timepoint
# Each row represents one volume — a value of 1 flags that volume as corrupted by large motion
!cat mo_regressors.txt

WM and CSF Signals

In [ ]:
# Change to subject directory
os.chdir('/content/ds005892/sub-MJF011')

# Run FAST to segment the structural image into CSF, GM and WM
# -b: output bias field, -B: apply bias field correction
!fast -b -B anat/sub-MJF011_T1w_brain.nii.gz

# Threshold tissue probability maps at 0.5 to create binary masks
# highres_brain_pve_0 = CSF, highres_brain_pve_1 = GM, highres_brain_pve_2 = WM
!fslmaths anat/sub-MJF011_T1w_brain_pve_0 -thr 0.5 -bin melodic_output.ica/CSF_thr
!fslmaths anat/sub-MJF011_T1w_brain_pve_2 -thr 0.5 -bin melodic_output.ica/WM_thr

# Change to melodic output directory
os.chdir('/content/ds005892/sub-MJF011/melodic_output.ica')

# Co-register CSF and WM masks from structural space into functional space
!flirt -in CSF_thr -ref reg/example_func -applyxfm -init reg/highres2example_func.mat -interp nearestneighbour -out EF_CSF_thr
!flirt -in WM_thr -ref reg/example_func -applyxfm -init reg/highres2example_func.mat -interp nearestneighbour -out EF_WM_thr

# Erode masks to avoid partial volume effects at tissue boundaries
!fslmaths EF_CSF_thr -kernel gauss 1.8 -ero -bin EF_CSF_ero
!fslmaths EF_WM_thr -kernel gauss 2.2 -ero -bin EF_WM_ero

# Extract mean timeseries from CSF and WM masks
!fslmeants -i filtered_func_data -m EF_CSF_ero -o CSF.txt
!fslmeants -i filtered_func_data -m EF_WM_ero -o WM.txt

In [ ]:
# Check the contents of the WM mean signal timeseries — one value per volume representing
# the average BOLD signal within the white matter mask at each timepoint
!cat WM.txt

Design Matrix for Nuisance Regressors

In [ ]:
# Load all regressors
mo_regressors = np.loadtxt('mo_regressors.txt')
ic_regressors = np.loadtxt('filtered_func_data.ica/melodic_mix')
rp_regressors = np.loadtxt('mc/prefiltered_func_data_mcf.par') # 6 MPs from MCFLIRT
csf_regressors = np.loadtxt('CSF.txt').reshape(-1, 1)  # reshape to column vector
wm_regressors = np.loadtxt('WM.txt').reshape(-1, 1)

# Combine all regressors into a single design matrix
regressors = np.hstack((ic_regressors, mo_regressors, rp_regressors, csf_regressors, wm_regressors))
np.savetxt('noise_regressors.txt', regressors)

# Identify noise columns: noise ICs + all motion outliers + realignment params + CSF + WM
with open('fix4melview_Standard_thr20.txt', 'r') as f:
    noise_ic_idx = f.readlines()[-1].translate(str.maketrans({"[": "", "]": "", "\n": ""})).split(", ")

# All columns after ICs are noise regressors
noise_mo_rp_csf_wm_idx = range(
    ic_regressors.shape[1] + 1,
    ic_regressors.shape[1] + mo_regressors.shape[1] + rp_regressors.shape[1] + 2 + 1
    # +2 for CSF and WM columns
)
noise_idx = ",".join([str(idx) for idx in np.hstack((noise_ic_idx, noise_mo_rp_csf_wm_idx))])

Regress out nuisance regressors

In [ ]:
!fsl_regfilt -i filtered_func_data -d noise_regressors.txt -o filtered_func_data_clean -f $noise_idx

Quality control: Nuisance Regression

In [ ]:
display(Markdown("### Raw Functional Image: Middle Volume, Brain"))

nv_FSL = AnyNiivue()
nv_FSL.load_volumes([{"path": "/content/ds005892/sub-MJF011/func/sub-MJF011_task-rest_bold.nii.gz"}])
nv_FSL

In [ ]:
display(Markdown("### Preprocessed Functional Image - Before nuisance regression"))

nv_FSL = AnyNiivue()
nv_FSL.load_volumes([{"path": "filtered_func_data.nii.gz"}])
nv_FSL


In [ ]:
display(Markdown("### Preprocessed Functional Image - After nuisance regression"))

nv_FSL = AnyNiivue()
nv_FSL.load_volumes([{"path": "filtered_func_data_clean.nii.gz"}])
nv_FSL


## Functional Connectivity Analysis

In [ ]:
import os
import matplotlib.pyplot as plt
from nilearn import datasets, maskers, connectome, plotting
os.chdir('/content/ds005892/sub-MJF011')

In [ ]:
# Load Schaefer atlas with 100 nodes
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7)
atlas_filename = atlas.maps
labels = atlas.labels[1:]  # Remove background label

In [ ]:
# Visualize the Schaefer 2018 parcellation (100 nodes) overlaid on the MNI152 T1 template

# Load MNI template as base, then overlay the atlas
display(Markdown("### Schaefer 2018 Atlas (100 nodes, Yeo 7 Networks)"))

nv_atlas = AnyNiivue()
nv_atlas.load_volumes([
    {   "path": "anat/standard_brain.nii.gz", "colormap": "gray"},
    {   "path": atlas.maps, "colormap": "random", "interpolation": "nearest",
       }
])
nv_atlas

In [ ]:
# Extract the average BOLD signal from each of the 100 regions (clean data)
masker = maskers.NiftiLabelsMasker(
    labels_img=atlas_filename,
    standardize='zscore_sample',
    memory='nilearn_cache',
    verbose=1
)

# Noisy (for comparison)
time_series_rest_noisy = masker.fit_transform('melodic_output.ica/filtered_func_data.nii.gz')
# Clean
time_series_rest_clean = masker.fit_transform('melodic_output.ica/filtered_func_data_clean.nii.gz')

In [ ]:
# Compute Pearson correlation between all pairs of regions (clean data)
correlation_measure = connectome.ConnectivityMeasure(kind='correlation', standardize='zscore_sample')

# Noisy
correlation_matrix_noisy = correlation_measure.fit_transform([time_series_rest_noisy])[0]

# Clean
correlation_matrix_clean = correlation_measure.fit_transform([time_series_rest_clean])[0]

Quality Control: Functional Connectivity Analysis

In [ ]:
# Plot the FC matrix - Noisy
plt.figure(figsize=(6, 4))
plt.imshow(correlation_matrix_noisy, interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('FC Matrix (Noisy)')
plt.colorbar(label='Pearson Correlation')
plt.xlabel('nodes')
plt.ylabel('nodes')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the FC matrix - Clean
plt.figure(figsize=(6, 4))
plt.imshow(correlation_matrix_clean, interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('FC Matrix (Clean)')
plt.colorbar(label='Pearson Correlation')
plt.xlabel('nodes')
plt.ylabel('nodes')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the connectivity matrix as a connectome on a brain

# Use only strong connections for clarity (threshold)
threshold = 0.6
adj_matrix = correlation_matrix_clean.copy()
adj_matrix[np.abs(adj_matrix) < threshold] = 0

# Get region coordinates from the atlas
coords = plotting.find_parcellation_cut_coords(labels_img=atlas_filename)

# Interactive 3D connectome view
view = plotting.view_connectome(
    adj_matrix,
    coords,
    edge_threshold="80%",
    node_size=10,
    colorbar=True
)
view

# Default Mode Network (DMN) Identification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

from IPython.display import display, Markdown
from ipyniivue import AnyNiivue
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi, show

### Obtain preprocessed fMRI files from previous step.

In [ ]:
import os

original_dir = '/content'
# Ensure we are in the base /content directory for consistent download location
os.chdir(original_dir)

if not os.path.exists('preprocessed_fmri.zip'):
    print('Downloading preprocessed fMRI. This should take less than 2 minutes.')
    print('Installing Gdown...')
    !pip install -q gdown
    !gdown "1_LRqGMDFfqzKwGK0C9Evvrf4zL4chr22" -O preprocessed_fmri.zip
    # Unzip directly to /content, assuming the zip contains the Neuroimaging2026/ structure
    !unzip -q -o preprocessed_fmri.zip -d /content/
    print("✓ Done")
else:
    print("✓ Already exists, skipping download.")

### Get Spatial IC Maps from MELODIC output.

In [ ]:
import os

# Ensure PATTERN2 and original_dir are available
original_dir = '/content'
PATTERN2 = 'sub-MJF011'

# The melodic_IC.nii.gz is now expected in /content/Neuroimaging2026/sub-MJF011/melodic_output.ica/
INPUT_FILE=os.path.join(original_dir, 'Neuroimaging2026', PATTERN2, 'melodic_output.ica', 'melodic_IC.nii.gz')
melodic_img = nib.load(INPUT_FILE)
melodic_img_data = melodic_img.get_fdata()

# Sanity-check voxel sizes and shape
print(melodic_img.header.get_zooms())
print(melodic_img_data.shape)

### Create Yeo DMN template

In [ ]:
yeo_dataset = datasets.fetch_atlas_yeo_2011()

In [ ]:
yeo_maps = nib.load(yeo_dataset['maps'])
yeo_maps_data = yeo_maps.get_fdata()

# Create a binary mask: 1 where the value is 7, and 0 everywhere else
dmn_mask_data = (yeo_maps_data == 7).astype(np.float32)

dmn_mask_img = nib.Nifti1Image(dmn_mask_data, yeo_maps.affine, yeo_maps.header)

dmn_mask_data_3d = np.squeeze(dmn_mask_data)

dmn_mask_img = nib.Nifti1Image(dmn_mask_data_3d, yeo_maps.affine)
# Save the DMN mask to a consistent, easily accessible absolute path in /content
nib.save(dmn_mask_img, os.path.join(original_dir, 'dmn_mask_yeo.nii.gz'))

In [ ]:
plot_roi(dmn_mask_img, title="Network 7 (Default Mode Network)")
show()

### Examine space of DMN mask / atlas

In [ ]:
print(dmn_mask_img.affine)
### MNI 1 mm RAS ###

### Ensure that the Yeo Mask and Spatial IC Maps are in the same MNI space

In [ ]:
# Ensure PATTERN2 and original_dir are available
# The current working directory should be /content/ds005892/sub-MJF011/melodic_output.ica/
original_dir = '/content'
PATTERN2 = 'sub-MJF011'

# Change to the melodic output directory where these files should be created
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN2, 'melodic_output.ica'))
print(f"Current working directory for melodic_img_mni generation: {os.getcwd()}")

# Step 1:
# Use transformation matrices from fMRI preprocessing step to bring input from MELODIC (i.e., spatial IC maps) into MNI space.
# (Note: MELODIC OUTPUTS are currently in functional space)
! convert_xfm -omat func2standard.mat \
            -concat {original_dir}/ds005892/{PATTERN2}/reg/struct2standard.mat \
                    {original_dir}/ds005892/{PATTERN2}/reg/rest2struct.mat

! flirt -in {INPUT_FILE} \
      -ref {original_dir}/ds005892/{PATTERN2}/anat/standard_brain.nii.gz \
      -applyxfm \
      -init func2standard.mat \
      -out melodic_img_mni.nii.gz

print("Files in current directory after melodic_img_mni generation:")
!ls
### OUTPUT: spatial IC maps in standard MNI 2 mm ###

In [ ]:
# Step 2:
# Resample DMN mask from MNI 1 mm to MNI 2 mm
# Reference the centrally saved dmn_mask_yeo.nii.gz and the melodic_img_mni.nii.gz from CWD

# Ensure we are in the correct directory where the flirt outputs should be
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN2, 'melodic_output.ica'))
print(f"Current working directory for dmn_mask_mni generation: {os.getcwd()}")
print("Files in current directory before dmn_mask_mni generation:")
!ls

! flirt -in {original_dir}/dmn_mask_yeo.nii.gz \
      -ref melodic_img_mni.nii.gz \
      -applyxfm -usesqform \
      -interp nearestneighbour \
      -out dmn_mask_mni.nii.gz

print("Files in current directory after dmn_mask_mni generation:")
!ls
### OUTPUT: DMN Mask in standard MNI 2 mm ###

In [ ]:
import os

# Ensure PATTERN2 and original_dir are available
original_dir = '/content'
PATTERN2 = 'sub-MJF011'

# Ensure we are in the correct directory where the melodic output files are located
os.chdir(os.path.join(original_dir, 'ds005892', PATTERN2, 'melodic_output.ica'))
print(f"Current working directory: {os.getcwd()}")
print("Files in current directory:")
!ls

# Load dmn_mask_mni.nii.gz and melodic_img_mni.nii.gz from the current directory
dmn_mask = nib.load('dmn_mask_mni.nii.gz')
melodic_img_mni = nib.load('melodic_img_mni.nii.gz')
assert np.array_equal(dmn_mask.affine, melodic_img_mni.affine)

### Binarize the IC spatial maps with cluster-defining threshold >= 2.3 *
*threshold from Woo et al., 2014 - Cluster-extent based thresholding in fMRI analyses: Pitfalls and recommendations

In [ ]:
melodic_img_mni_data = melodic_img_mni.get_fdata()
binary_ic_maps = (melodic_img_mni.get_fdata() > 2.3).astype(np.int8)

### Extract best-matching IC map with DMN mask using Dice coefficient

In [ ]:
def dice_coefficient(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    intersection = np.logical_and(a, b).sum()
    return 2 * intersection / (a.sum() + b.sum())

dice_scores = np.array([
    dice_coefficient(binary_ic_maps[..., i], dmn_mask.get_fdata())
    for i in range(binary_ic_maps.shape[-1])
])

best_ic_index = np.argmax(dice_scores)

print(f"Best IC index: {best_ic_index}, Dice: {dice_scores[best_ic_index]:.4f}")

### Obtain average Z-score within DMN mask

In [ ]:
best_ic = melodic_img_mni_data[..., best_ic_index]
mean_val = best_ic[dmn_mask.get_fdata() > 0].mean()
print(f'Average Z-score within DMN mask: {mean_val}')

### Visualize best-matching IC map.

In [ ]:
best_ic_pos = np.clip(best_ic, 0, None)
best_ic_img = nib.Nifti1Image(best_ic_pos, affine=melodic_img_mni.affine)

# Corrected path for standard_brain.nii.gz
standard_brain = nib.load(f'{original_dir}/ds005892/{PATTERN2}/anat/standard_brain.nii.gz')
title = f'Best-matching IC - Subject 11 (Dice: {dice_scores[best_ic_index]:.2f}, Mean Z-score: {mean_val:.2f})'
plotting.plot_stat_map(
    best_ic_img,
    bg_img=standard_brain,
    display_mode='ortho',
    cut_coords=(8, -66, 22),
    colorbar=True,
    cmap='hot',
    threshold=2,
    vmax=5,
    title=title
)
plotting.show()

# Volume comparison

### Visualizing MJF011's Putamen (blue) on MJF008's Putamen (red)

In [ ]:
MJF011_on_MJF008_putamen = AnyNiivue()
volumes = [
    {"path": "../../sub-MJF008/anat/MJF008_putamen_L_Puta.nii.gz", "colormap": "red", "opacity": 0.7},
    {"path": "../../sub-MJF008/anat/MJF008_putamen_R_Puta.nii.gz", "colormap": "red", "opacity": 0.7},
    {"path": "../anat/MJF011_putamen_L_Puta.nii.gz", "colormap": "blue", "opacity": 0.7},
    {"path": "../anat/MJF011_putamen_R_Puta.nii.gz", "colormap": "blue", "opacity": 0.7}
]
MJF011_on_MJF008_putamen.load_volumes(volumes)
MJF011_on_MJF008_putamen

### Visualizing MJF011's Hippocampus (blue) on MJF008's Hippocampus (red)

In [ ]:
MJF011_on_MJF008_hippocampus = AnyNiivue()
volumes = [
    {"path": "../../sub-MJF008/anat/MJF008_hippocampus_L_Hipp.nii.gz", "colormap": "red", "opacity": 0.7},
    {"path": "../../sub-MJF008/anat/MJF008_hippocampus_R_Hipp.nii.gz", "colormap": "red", "opacity": 0.7},
    {"path": "../anat/MJF011_hippocampus_L_Hipp.nii.gz", "colormap": "blue", "opacity": 0.7},
    {"path": "../anat/MJF011_hippocampus_R_Hipp.nii.gz", "colormap": "blue", "opacity": 0.7}
]
MJF011_on_MJF008_hippocampus.load_volumes(volumes)
MJF011_on_MJF008_hippocampus

### Visualizing MJF011's Amygdala (blue) on MJF008's Amygdala (red)

In [ ]:
MJF011_on_MJF008_amygdala = AnyNiivue()
volumes = [
    {"path": "../../sub-MJF008/anat/MJF008_amygdala_L_Amyg.nii.gz", "colormap": "red", "opacity": 0.7},
    {"path": "../../sub-MJF008/anat/MJF008_amygdala_R_Amyg.nii.gz", "colormap": "red", "opacity": 0.7},
    {"path": "../anat/MJF011_amygdala_L_Amyg.nii.gz", "colormap": "blue", "opacity": 0.7},
    {"path": "../anat/MJF011_amygdala_R_Amyg.nii.gz", "colormap": "blue", "opacity": 0.7}
]
MJF011_on_MJF008_amygdala.load_volumes(volumes)
MJF011_on_MJF008_amygdala

# Discussion


In this study, we observed volume reduction in the putamen, amygdala, and hippocampus, along with decreased connectivity in the DMN in the PD-NC subject compared to the HC subject. These findings are consistent with known structural and functional alterations associated with PD, affecting both motor and cognitive-related brain regions.

However, it is important to highlight that this analysis was conducted on only two subjects. As a result, no generalizable conclusions can be drawn, and the findings should be interpreted with caution. The observed differences may reflect individual variability rather than disease-specific patterns, and therefore remain largely speculative.

Future work involving a larger cohort would be necessary to validate these observations and to better characterize the relationship between brain morphometry, functional connectivity, and clinical symptoms in PD.